# 02 - Feature Engineering V15

This notebook preserves the thesis-facing feature-engineering pipeline for the Home Credit Default Risk project. It starts with modular script-based feature generation, then shows applicant-level merging, progressive feature extensions, stability filtering, and final V15 construction.

Raw Kaggle CSV files and full generated parquet matrices are local-only artifacts and are not committed to GitHub. Heavy cells are retained for traceability but should not be run during a video demo unless the local data/artifacts are available.


## Thesis terminology

- Final thesis-facing feature set: **V15**.
- SPC = **Stacked Prediction Candidate** = implementation name `fp_final`.
- V15 Multi-Seed LightGBM = implementation name `lgb_v15_ms`.
- SE-HC = **Stacked Ensemble with Hill-Climbing Selection** = implementation name `SIGMA_FINAL`.
- Final thesis-facing model formula: `SE-HC = 0.5 * SPC + 0.5 * V15 Multi-Seed LightGBM`.
- V16, BLOCK_A_FINAL, and selected-candidate SE-HC variants are archived/post-thesis experiments and are not final thesis feature/model logic here.


## Section 1 - Feature engineering overview

Home Credit is a relational tabular dataset. The project uses modular `.py` scripts for early feature generation because each relational table requires table-specific aggregation logic before it can be joined to the applicant table.

Pipeline flow:

```text
Raw Home Credit tables
-> script-based relational feature generation
-> applicant-level feature tables
-> merge into train/test application tables
-> application preprocessing and rebuild
-> advanced behavioral / interaction / auxiliary features
-> stability filtering
-> final V15 target-score features
-> final V15 matrices
```

All relational tables are aggregated to applicant level using `SK_ID_CURR`. V15 is the final thesis feature representation, with **1,043 columns including `SK_ID_CURR` and `TARGET`**, corresponding to **1,041 modelling features** after excluding ID and target columns.


## Section 2 - Script-based relational feature generation

The early feature layer is implemented through scripts in `feature_engineering/`. These cells are heavy because they read raw Kaggle tables and aggregate them into applicant-level feature files. Do not run during the demo unless raw data and sufficient runtime are available.


In [ ]:

from pathlib import Path

FEATURE_SCRIPT_DIR = Path('feature_engineering')
feature_scripts = [
    'feature - bureau & bureau balance.py',
    'feature - previous_application.py',
    'feature - installments_payments.py',
    'feature - POS_CASH_balance.py',
    'feature - credit_card_balance.py',
    'feature - clustering.py',
    'feature - trend.py',
]

for script in feature_scripts:
    path = FEATURE_SCRIPT_DIR / script
    print(f'{path} | exists={path.exists()} | size={path.stat().st_size if path.exists() else "missing"}')


feature_engineering\feature - bureau & bureau balance.py | exists=True | size=10922
feature_engineering\feature - previous_application.py | exists=True | size=10311
feature_engineering\feature - installments_payments.py | exists=True | size=10800
feature_engineering\feature - POS_CASH_balance.py | exists=True | size=5823
feature_engineering\feature - credit_card_balance.py | exists=True | size=4670
feature_engineering\feature - clustering.py | exists=True | size=1541
feature_engineering\feature - trend.py | exists=True | size=8041


### `feature - bureau & bureau balance.py`

Aggregates external bureau credit history and monthly bureau balance. Captures number of loans, active/closed loans, overdue behavior, credit limits, debt burden, and historical delinquency.


In [ ]:
%run "feature_engineering/feature - bureau & bureau balance.py"


### `feature - previous_application.py`

Aggregates previous Home Credit applications. Captures approval/refusal history, previous loan amounts, annuity, interest-related proxies, product/channel information, and historical application behavior.


In [ ]:
%run "feature_engineering/feature - previous_application.py"


### `feature - installments_payments.py`

Aggregates repayment behavior. Captures late payment, early payment, underpayment, overpayment, days past due, payment ratio, and repayment consistency.


In [ ]:
%run "feature_engineering/feature - installments_payments.py"


### `feature - POS_CASH_balance.py`

Aggregates POS cash loan monthly status. Captures delinquency status, remaining installments, contract status dynamics, and recent repayment behavior.


In [ ]:
%run "feature_engineering/feature - POS_CASH_balance.py"


### `feature - credit_card_balance.py`

Aggregates credit card balance behavior. Captures utilization, drawing behavior, payment behavior, balance, credit limit usage, and delinquency indicators.


In [ ]:
%run "feature_engineering/feature - credit_card_balance.py"


### `feature - clustering.py`

Creates clustering-based applicant or behavior segments. These segments are used as additional high-level segmentation features.


In [ ]:
%run "feature_engineering/feature - clustering.py"


### `feature - trend.py`

Creates trend or temporal-change features from historical behavior. These features help capture whether repayment or balance behavior is improving or worsening over time.


In [ ]:
%run "feature_engineering/feature - trend.py"


## Section 3 - Applicant-level merge and validation

All relational features must be aggregated to one row per applicant before being merged by `SK_ID_CURR`. Duplicate checks are necessary to prevent many-to-one merge errors.

This section loads generated feature tables, checks shape and duplicate keys, inspects missingness where useful, merges all feature tables into `application_train`/`application_test`, and saves the initial merged matrices.


In [ ]:

from pathlib import Path

candidate_outputs = [
    Path('bureau_feature.csv'),
    Path('ip_feature.csv'),
    Path('pa_feature.csv'),
    Path('pcb_feature.csv'),
    Path('ccb_feature.csv'),
    Path('processed_train_test/trend_features.parquet'),
    Path('processed_train_test/clustering_features.parquet'),
    Path('feature_engineering/fe_v2/bureau_deep.parquet'),
    Path('feature_engineering/fe_v2/prev_detailed.parquet'),
    Path('feature_engineering/fe_v2/inst_behavior.parquet'),
    Path('feature_engineering/fe_v2/cross_features.parquet'),
]

for path in candidate_outputs:
    print(f'{path} | exists={path.exists()} | size={path.stat().st_size if path.exists() else "local-only/missing"}')


bureau_feature.csv | exists=False | size=local-only/missing
ip_feature.csv | exists=False | size=local-only/missing
pa_feature.csv | exists=False | size=local-only/missing
pcb_feature.csv | exists=False | size=local-only/missing
ccb_feature.csv | exists=False | size=local-only/missing
processed_train_test\trend_features.parquet | exists=True | size=2793713
processed_train_test\clustering_features.parquet | exists=True | size=7072745
feature_engineering\fe_v2\bureau_deep.parquet | exists=True | size=30185700
feature_engineering\fe_v2\prev_detailed.parquet | exists=True | size=18321595
feature_engineering\fe_v2\inst_behavior.parquet | exists=True | size=17252849
feature_engineering\fe_v2\cross_features.parquet | exists=True | size=24746350


In [ ]:
import pandas as pd

bureau_feat = pd.read_csv("bureau_feature.csv")

bureau_feat.head()

FileNotFoundError: [Errno 2] No such file or directory: 'bureau_feature.csv'

In [ ]:
bureau_feat.shape

In [ ]:
bureau_feat["SK_ID_CURR"].duplicated().sum()

In [ ]:
bureau_feat.isnull().mean().sort_values(ascending=False).head(20)

In [ ]:
bureau_feat.info(memory_usage="deep")

In [ ]:
bureau_feat.to_parquet(
    "bureau_feature.parquet",
    index=False
)

In [ ]:
import pandas as pd

bureau_feat = pd.read_csv("bureau_feature.csv")
ip_feat = pd.read_csv("ip_feature.csv")
pa_feat = pd.read_csv("pa_feature.csv")
pcb_feat = pd.read_csv("pcb_feature.csv")
ccb_feat = pd.read_csv("ccb_feature.csv")

In [ ]:
print(bureau_feat.shape)
print(ip_feat.shape)
print(pa_feat.shape)
print(pcb_feat.shape)
print(ccb_feat.shape)

In [ ]:
for df, name in [
    (bureau_feat, "bureau"),
    (ip_feat, "installments"),
    (pa_feat, "previous_application"),
    (pcb_feat, "pos_cash"),
    (ccb_feat, "credit_card")
]:

    print(f"\n{name}")

    print(
        "Duplicate SK_ID_CURR:",
        df["SK_ID_CURR"].duplicated().sum()
    )

In [ ]:
from pathlib import Path

RAW_DATA_DIR = Path('data/raw')  # GitHub excludes raw Kaggle CSVs; create this locally after download.
train = pd.read_csv(RAW_DATA_DIR / 'application_train.csv')
test = pd.read_csv(RAW_DATA_DIR / 'application_test.csv')


In [ ]:
feature_tables = [
    bureau_feat,
    ip_feat,
    pa_feat,
    pcb_feat,
    ccb_feat
]

for feat in feature_tables:

    train = train.merge(
        feat,
        on="SK_ID_CURR",
        how="left"
    )

    test = test.merge(
        feat,
        on="SK_ID_CURR",
        how="left"
    )

In [ ]:
print(train.shape)
print(test.shape)

(307511, 409)
(48744, 408)


In [ ]:
train.to_parquet(
    "train_merged.parquet",
    index=False
)

test.to_parquet(
    "test_merged.parquet",
    index=False
)

## Section 4 - Application preprocessing and rebuild

This stage handles application-level anomalies and sentinel values, standardizes basic ratios, and rebuilds the merged train/test matrices so downstream feature versions share a consistent applicant table.


### **Preprocessing app_train**


In [ ]:
"""
preprocess_application.py
Engineer features từ application_train.csv + application_test.csv.
Output: application_processed.parquet (1 row per SK_ID_CURR)
"""
import numpy as np
import pandas as pd

DATA_DIR = 'data/raw'   # adjust


def preprocess_application():
    print('=' * 60)
    print('Preprocessing application_train/test')
    print('=' * 60)

    train = pd.read_csv(f'{DATA_DIR}/application_train.csv')
    test  = pd.read_csv(f'{DATA_DIR}/application_test.csv')
    print(f'  Train: {train.shape}, Test: {test.shape}')

    # Concat để engineer cùng nhau
    df = pd.concat([train, test], ignore_index=True, sort=False)

    # === NHÓM 1: Cleaning ===
    print('\n[Nhóm 1] Cleaning...')

    # Sentinels
    df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
    df['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan, inplace=True)
    print(f'  DAYS_EMPLOYED max after sentinel fix: {df["DAYS_EMPLOYED"].max():.0f}')

    # Anomalous CODE_GENDER
    n_before = len(df)
    df = df[df['CODE_GENDER'] != 'XNA'].reset_index(drop=True)
    print(f'  Removed {n_before - len(df)} XNA rows')

    # === NHÓM 2: EXT_SOURCE engineering ===
    print('\n[Nhóm 2] EXT_SOURCE features...')

    ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

    # Statistics
    df['EXT_MEAN']     = df[ext].mean(axis=1)
    df['EXT_STD']      = df[ext].std(axis=1)
    df['EXT_MIN']      = df[ext].min(axis=1)
    df['EXT_MAX']      = df[ext].max(axis=1)
    df['EXT_MEDIAN']   = df[ext].median(axis=1)
    df['EXT_PROD']     = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_WEIGHTED'] = (df['EXT_SOURCE_1']*2 + df['EXT_SOURCE_2']*3 + df['EXT_SOURCE_3']*4)

    # Fill EXT_STD NaN với mean (vì std cần >=2 non-null values)
    df['EXT_STD'].fillna(df['EXT_STD'].mean(), inplace=True)

    # Pairwise products
    df['EXT_1x2'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2']
    df['EXT_2x3'] = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_1x3'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_3']

    # Interactions với DAYS_*
    for s in ext:
        df[f'{s}_x_DAYS_BIRTH']    = df[s] * df['DAYS_BIRTH']
        df[f'{s}_x_DAYS_EMPLOYED'] = df[s] * df['DAYS_EMPLOYED']

    # Null indicators
    df['EXT_NULL_COUNT'] = df[ext].isnull().sum(axis=1)
    for s in ext:
        df[f'{s}_NULL'] = df[s].isnull().astype(np.int8)

    print(f'  Added {sum(["EXT" in c for c in df.columns])} EXT-related features')

    # === NHÓM 3: Financial ratios ===
    print('\n[Nhóm 3] Financial ratios...')

    df['CREDIT_INCOME_RATIO']  = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['CREDIT_TERM']          = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['INCOME_PER_PERSON']    = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    df['PAYMENT_RATE']         = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['INCOME_CREDIT_PERC']   = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']

    df['CREDIT_GOODS_RATIO']   = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
    df['CREDIT_GOODS_DIFF']    = df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']
    df['GOODS_INCOME_RATIO']   = df['AMT_GOODS_PRICE'] / df['AMT_INCOME_TOTAL']

    df['DAYS_EMPLOYED_PERC']   = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['INCOME_PER_AGE']       = df['AMT_INCOME_TOTAL'] / df['DAYS_BIRTH']
    df['CAR_BIRTH_RATIO']      = df['OWN_CAR_AGE'] / df['DAYS_BIRTH']
    df['CAR_EMPLOYED_RATIO']   = df['OWN_CAR_AGE'] / df['DAYS_EMPLOYED']
    df['CHILDREN_RATIO']       = df['CNT_CHILDREN'] / df['CNT_FAM_MEMBERS']
    df['PHONE_BIRTH_RATIO']    = df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH']

    print(f'  Added 15 ratio features')

    # === NHÓM 4: Missingness signals ===
    print('\n[Nhóm 4] Missingness signals...')

    # Tổng missing per row
    df['MISSING_COUNT'] = df.isnull().sum(axis=1)

    # Document flag sum
    doc_cols = [c for c in df.columns if 'FLAG_DOCUMENT' in c]
    df['DOCUMENT_COUNT'] = df[doc_cols].sum(axis=1)
    print(f'  Document columns counted: {len(doc_cols)}')

    # Bureau request total
    amt_req_cols = [c for c in df.columns if 'AMT_REQ_CREDIT_BUREAU' in c]
    df['AMT_REQ_TOTAL'] = df[amt_req_cols].sum(axis=1)

    # Null flags
    null_flag_cols = ['OWN_CAR_AGE', 'OCCUPATION_TYPE',
                      'AMT_REQ_CREDIT_BUREAU_HOUR']
    for c in null_flag_cols:
        if c in df.columns:
            df[f'{c}_NULL'] = df[c].isnull().astype(np.int8)

    # === Final cleanup ===
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    print(f'\nFinal shape: {df.shape}')
    print(f'New features engineered: ~{df.shape[1] - 122}')

    # Save
    df.to_parquet('./application_processed.parquet')
    print('Saved -> application_processed.parquet')

    return df


if __name__ == '__main__':
    preprocess_application()

Preprocessing application_train/test
  Train: (307511, 122), Test: (48744, 121)

[Nhóm 1] Cleaning...
  DAYS_EMPLOYED max after sentinel fix: 0
  Removed 4 XNA rows

[Nhóm 2] EXT_SOURCE features...
  Added 23 EXT-related features

[Nhóm 3] Financial ratios...
  Added 15 ratio features

[Nhóm 4] Missingness signals...
  Document columns counted: 20

Final shape: (356251, 163)
New features engineered: ~41
Saved -> application_processed.parquet


In [ ]:
"""rebuild_merged.py"""
import pandas as pd

# Load processed application
app = pd.read_parquet('./application_processed.parquet')

# Load tất cả 7 feature tables
bureau_feat = pd.read_csv("bureau_feature.csv")
ip_feat     = pd.read_csv("ip_feature.csv")
pa_feat     = pd.read_csv("pa_feature.csv")
pcb_feat    = pd.read_csv("pcb_feature.csv")
ccb_feat    = pd.read_csv("ccb_feature.csv")
trend       = pd.read_parquet('./trend_features.parquet').reset_index()
clu         = pd.read_parquet('./clustering_features.parquet').reset_index()

trend.rename(columns={'index': 'SK_ID_CURR'}, inplace=True)
clu.rename(columns={'index': 'SK_ID_CURR'}, inplace=True)

# Merge all
for feat in [bureau_feat, ip_feat, pa_feat, pcb_feat, ccb_feat, trend, clu]:
    app = app.merge(feat, on='SK_ID_CURR', how='left')

# Split train/test
train = app[app['TARGET'].notnull()].reset_index(drop=True)
test  = app[app['TARGET'].isnull()].reset_index(drop=True)
test.drop(columns='TARGET', inplace=True)

print(f'train: {train.shape}')
print(f'test:  {test.shape}')

train.to_parquet('./train_merged_v6.parquet')
test.to_parquet('./test_merged_v6.parquet')

train: (307507, 501)
test:  (48744, 500)


## Section 5 - Advanced feature extensions

The following sections are progressive feature versions built on top of the initial merged matrix. They are feature-representation changes, not final model changes. The final thesis-facing feature representation remains V15.


### Clustering, trend, and extra feature merge

Requires local feature scripts/artifacts; kept for pipeline traceability. This stage adds intermediate trend and clustering outputs to the merged applicant-level matrix.


In [ ]:
import pandas as pd
train = pd.read_parquet('train_merged.parquet')

# 1. HOUR_APPR_PROCESS_START — phải là categorical
print('HOUR_APPR_PROCESS_START:')
print(f'  dtype: {train["HOUR_APPR_PROCESS_START"].dtype}')
print(f'  nunique: {train["HOUR_APPR_PROCESS_START"].nunique()}')
print(f'  values: {sorted(train["HOUR_APPR_PROCESS_START"].dropna().unique())[:5]}...')

# 2. Check OCCUPATION_TYPE
print('\nOCCUPATION_TYPE:')
print(f'  dtype: {train["OCCUPATION_TYPE"].dtype}')
print(f'  nunique: {train["OCCUPATION_TYPE"].nunique()}')

# 3. Verify cat_features detection
from __main__ import detect_categorical_columns, clean_feature_names
train = clean_feature_names(train)
cat = detect_categorical_columns(train)
print(f'\n#Detected categorical: {len(cat)}')
print(f'HOUR_APPR_PROCESS_START in cat: {"HOUR_APPR_PROCESS_START" in cat}')
print(f'OCCUPATION_TYPE in cat: {"OCCUPATION_TYPE" in cat}')

HOUR_APPR_PROCESS_START:
  dtype: int64
  nunique: 24
  values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]...

OCCUPATION_TYPE:
  dtype: object
  nunique: 18

#Detected categorical: 36
HOUR_APPR_PROCESS_START in cat: True
OCCUPATION_TYPE in cat: False


### **FE 02: Xử lý thêm clustering và trend features**

In [ ]:
import gc
import numpy as np
import pandas as pd
from typing import List, Tuple
from contextlib import contextmanager
import time


@contextmanager
def timer(name):
    t0 = time.time(); print(f'[{name}] start'); yield
    print(f'[{name}] done in {time.time()-t0:.1f}s')


# ============================================================================
# CORE TREND FUNCTIONS
# ============================================================================
def _slope(group: np.ndarray, time: np.ndarray) -> float:
    """
    Linear regression slope of `group` against `time`.
    Returns NaN if fewer than 2 valid points.
    """
    mask = ~np.isnan(group) & ~np.isnan(time)
    if mask.sum() < 2:
        return np.nan
    x = time[mask]
    y = group[mask]
    if x.std() == 0:
        return 0.0
    # numpy polyfit deg=1 returns [slope, intercept]
    return np.polyfit(x, y, 1)[0]


def _trend_features_per_group(df: pd.DataFrame,
                              group_key: str,
                              value_cols: List[str],
                              time_col: str,
                              recent_days: int = 180,
                              prefix: str = 'TREND'
                              ) -> pd.DataFrame:
    """
    For each group, compute trend stats for each value column:
      - slope over time
      - recent_mean - older_mean (delta)

    Notes
    -----
    - Uses raw groupby + numpy for speed; pandas .apply with custom slope
      is 10x slower for this dataset size.
    - `time_col` is expected in days-since-application convention (negative
      means past).
    """
    # Filter for points with valid time
    df = df[df[time_col].notnull()].copy()

    # Recent / older flag
    df['_is_recent'] = (df[time_col] >= -recent_days).astype(np.int8)

    out = []

    for col in value_cols:
        sub = df[[group_key, time_col, col, '_is_recent']].copy()
        sub = sub[sub[col].notnull()]

        # 1) Slope per group
        slopes = (sub.groupby(group_key)
                     .apply(lambda g: _slope(g[col].values, g[time_col].values))
                     .rename(f'{prefix}_{col}_SLOPE'))

        # 2) Recent mean - older mean
        recent_mean = (sub[sub['_is_recent'] == 1]
                       .groupby(group_key)[col].mean()
                       .rename(f'{prefix}_{col}_RECENT_MEAN'))
        older_mean = (sub[sub['_is_recent'] == 0]
                      .groupby(group_key)[col].mean()
                      .rename(f'{prefix}_{col}_OLDER_MEAN'))

        delta = (recent_mean.fillna(0) - older_mean.fillna(0)
                ).rename(f'{prefix}_{col}_DELTA_RECENT_OLDER')

        out += [slopes, recent_mean, older_mean, delta]

    result = pd.concat(out, axis=1)
    return result


# ============================================================================
# 1. INSTALLMENTS TREND
# ============================================================================
def installments_trend(data_dir: str) -> pd.DataFrame:
    """
    Trend features from installments_payments.

    High-signal trends:
      - DPD (days past due)            -> slope >0 means worsening behaviour
      - PAYMENT_PERC                    -> declining payment ratio is bad
      - PAYMENT_DIFF                    -> growing under-payment is bad
    """
    ins = pd.read_csv(f'{data_dir}/installments_payments.csv')

    ins['DPD']           = (ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']).clip(lower=0)
    ins['PAYMENT_PERC']  = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['PAYMENT_DIFF']  = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    ins.replace([np.inf, -np.inf], np.nan, inplace=True)

    feats = _trend_features_per_group(
        ins,
        group_key='SK_ID_CURR',
        value_cols=['DPD', 'PAYMENT_PERC', 'PAYMENT_DIFF'],
        time_col='DAYS_INSTALMENT',
        recent_days=365,
        prefix='INSTAL_TREND',
    )

    del ins; gc.collect()
    return feats


# ============================================================================
# 2. CREDIT CARD TREND
# ============================================================================
def credit_card_trend(data_dir: str) -> pd.DataFrame:
    """
    Trend features from credit_card_balance.

    Months balance is in MONTHS_BALANCE (not days). We convert to days for
    consistency: days = months * 30.
    """
    cc = pd.read_csv(f'{data_dir}/credit_card_balance.csv')

    cc['CC_UTILIZATION'] = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL']
    cc.replace([np.inf, -np.inf], np.nan, inplace=True)
    cc['_DAYS'] = cc['MONTHS_BALANCE'] * 30   # negative = past

    feats = _trend_features_per_group(
        cc,
        group_key='SK_ID_CURR',
        value_cols=['AMT_BALANCE', 'CC_UTILIZATION', 'AMT_DRAWINGS_CURRENT'],
        time_col='_DAYS',
        recent_days=365,
        prefix='CC_TREND',
    )

    del cc; gc.collect()
    return feats


# ============================================================================
# 3. POS_CASH TREND
# ============================================================================
def pos_cash_trend(data_dir: str) -> pd.DataFrame:
    pos = pd.read_csv(f'{data_dir}/POS_CASH_balance.csv')
    pos['_DAYS'] = pos['MONTHS_BALANCE'] * 30

    feats = _trend_features_per_group(
        pos,
        group_key='SK_ID_CURR',
        value_cols=['SK_DPD', 'SK_DPD_DEF', 'CNT_INSTALMENT_FUTURE'],
        time_col='_DAYS',
        recent_days=365,
        prefix='POS_TREND',
    )

    del pos; gc.collect()
    return feats


# 4. BUREAU_BALANCE TREND (status worsening?)

def bureau_balance_trend(data_dir: str) -> pd.DataFrame:
    """
    Aggregate bureau_balance status to SK_ID_CURR via SK_ID_BUREAU bridge.

    STATUS encodes payment status: '0' (good), '1'-'5' (DPD buckets), 'C', 'X'.
    We map to integer DPD severity, then compute trend.
    """
    bb = pd.read_csv(f'{data_dir}/bureau_balance.csv')
    bureau = pd.read_csv(f'{data_dir}/bureau.csv', usecols=['SK_ID_BUREAU', 'SK_ID_CURR'])

    # Map STATUS -> severity score
    status_map = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, 'C': 0, 'X': np.nan}
    bb['STATUS_NUM'] = bb['STATUS'].map(status_map)

    # Bridge to SK_ID_CURR
    bb = bb.merge(bureau, on='SK_ID_BUREAU', how='left')
    bb = bb[bb['SK_ID_CURR'].notnull()]
    bb['_DAYS'] = bb['MONTHS_BALANCE'] * 30

    # Aggregate first per SK_ID_CURR (combining all loans)
    feats = _trend_features_per_group(
        bb,
        group_key='SK_ID_CURR',
        value_cols=['STATUS_NUM'],
        time_col='_DAYS',
        recent_days=365,
        prefix='BB_TREND',
    )
    feats.index = feats.index.astype(int)

    del bb, bureau; gc.collect()
    return feats


# MAIN
def build_trend_features(data_dir: str) -> pd.DataFrame:
    """
    Build all trend features and merge into a single DataFrame indexed by
    SK_ID_CURR.
    """
    print('=' * 60)
    print('Trend Features')
    print('=' * 60)

    with timer('installments trend'):
        ins_trend = installments_trend(data_dir)
        print(f'  shape: {ins_trend.shape}')

    with timer('credit card trend'):
        cc_trend = credit_card_trend(data_dir)
        print(f'  shape: {cc_trend.shape}')

    with timer('pos_cash trend'):
        pos_trend = pos_cash_trend(data_dir)
        print(f'  shape: {pos_trend.shape}')

    with timer('bureau_balance trend'):
        bb_trend = bureau_balance_trend(data_dir)
        print(f'  shape: {bb_trend.shape}')

    # Merge all on index = SK_ID_CURR
    trend = (ins_trend
             .join(cc_trend, how='outer')
             .join(pos_trend, how='outer')
             .join(bb_trend, how='outer'))

    # Downcast
    for c in trend.columns:
        trend[c] = trend[c].astype(np.float32)

    print(f'\nFinal trend features: {trend.shape}')
    print(f'Memory: {trend.memory_usage(deep=True).sum()/1024**2:.1f} MB')

    return trend


if __name__ == '__main__':
    DATA_DIR = 'data/raw'
    trend = build_trend_features(DATA_DIR)
    trend.to_parquet('./trend_features.parquet')
    print('Saved -> ./trend_features.parquet')

Trend Features
[installments trend] start
  shape: (339578, 12)
[installments trend] done in 271.0s
[credit card trend] start
  shape: (103558, 12)
[credit card trend] done in 86.5s
[pos_cash trend] start
  shape: (337252, 12)
[pos_cash trend] done in 255.1s
[bureau_balance trend] start
  shape: (131725, 4)
[bureau_balance trend] done in 55.0s

Final trend features: (343907, 40)
Memory: 55.1 MB
Saved -> ./trend_features.parquet


In [ ]:
import pandas as pd
import numpy as np

trend = pd.read_parquet('./trend_features.parquet')

# 1. Coverage per column
print('Non-null ratio:')
print((trend.notna().sum() / len(trend)).round(3).to_string())

# 2. Variance check — feature constant là useless
print('\nStd per column:')
print(trend.std().round(4).to_string())

# 3. Top variance features (potential strong predictors)
print('\nTop 10 features by std (likely most signal):')
print(trend.std().sort_values(ascending=False).head(10).to_string())

Non-null ratio:
INSTAL_TREND_DPD_SLOPE                                0.985
INSTAL_TREND_DPD_RECENT_MEAN                          0.735
INSTAL_TREND_DPD_OLDER_MEAN                           0.904
INSTAL_TREND_DPD_DELTA_RECENT_OLDER                   0.651
INSTAL_TREND_PAYMENT_PERC_SLOPE                       0.985
INSTAL_TREND_PAYMENT_PERC_RECENT_MEAN                 0.735
INSTAL_TREND_PAYMENT_PERC_OLDER_MEAN                  0.904
INSTAL_TREND_PAYMENT_PERC_DELTA_RECENT_OLDER          0.651
INSTAL_TREND_PAYMENT_DIFF_SLOPE                       0.985
INSTAL_TREND_PAYMENT_DIFF_RECENT_MEAN                 0.735
INSTAL_TREND_PAYMENT_DIFF_OLDER_MEAN                  0.904
INSTAL_TREND_PAYMENT_DIFF_DELTA_RECENT_OLDER          0.651
CC_TREND_AMT_BALANCE_SLOPE                            0.299
CC_TREND_AMT_BALANCE_RECENT_MEAN                      0.301
CC_TREND_AMT_BALANCE_OLDER_MEAN                       0.207
CC_TREND_AMT_BALANCE_DELTA_RECENT_OLDER               0.207
CC_TREND_CC_UTILIZATION_

In [ ]:
import gc
import numpy as np
import pandas as pd
from typing import List
from contextlib import contextmanager
import time

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


@contextmanager
def timer(name):
    t0 = time.time(); print(f'[{name}] start'); yield
    print(f'[{name}] done in {time.time()-t0:.1f}s')


# ============================================================================
# HELPERS
# ============================================================================
LOG_COLS_HINTS = ['AMT_', 'CNT_', '_COUNT', '_SUM', 'DURATION']


def _is_log_col(col_name: str) -> bool:
    """Heuristic: log-transform columns that are amounts, counts, or sums."""
    return any(h in col_name for h in LOG_COLS_HINTS)


def _preprocess_for_clustering(features: pd.DataFrame,
                               clip_pct: float = 0.99
                               ) -> np.ndarray:
    """
    Robust preprocessing pipeline for clustering:
      1. log1p transform skewed columns (amounts, counts)
      2. Median imputation
      3. Standard scaling
      4. Clip outliers at +/- z-score corresponding to clip_pct
    """
    df = features.copy()

    # 1. Log-transform skewed columns
    for col in df.columns:
        if _is_log_col(col):
            # log1p handles 0 and small negative gracefully via abs
            vals = df[col].values
            df[col] = np.sign(vals) * np.log1p(np.abs(vals))

    # 2. Impute
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(df.values)

    # 3. Scale
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # 4. Clip outliers (99th percentile of absolute z-score)
    clip_val = np.percentile(np.abs(X), clip_pct * 100)
    X = np.clip(X, -clip_val, clip_val)

    return X, imputer, scaler


# ============================================================================
# 1. AGGREGATES (unchanged)
# ============================================================================
def _bureau_aggregates(data_dir: str) -> pd.DataFrame:
    bureau = pd.read_csv(f'{data_dir}/bureau.csv')

    bureau['CREDIT_DURATION']    = -bureau['DAYS_CREDIT'] + bureau['DAYS_CREDIT_ENDDATE']
    bureau['DEBT_CREDIT_RATIO']  = bureau['AMT_CREDIT_SUM_DEBT'] / bureau['AMT_CREDIT_SUM']
    bureau.replace([np.inf, -np.inf], np.nan, inplace=True)

    aggs = {
        'DAYS_CREDIT':            ['min', 'max', 'mean'],
        'DAYS_CREDIT_ENDDATE':    ['mean'],
        'AMT_CREDIT_SUM':         ['mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT':    ['mean', 'sum'],
        'AMT_CREDIT_SUM_OVERDUE': ['mean', 'max'],
        'CREDIT_DURATION':        ['mean'],
        'DEBT_CREDIT_RATIO':      ['mean', 'max'],
        'CNT_CREDIT_PROLONG':     ['sum'],
    }
    agg = bureau.groupby('SK_ID_CURR').agg(aggs)
    agg.columns = pd.Index([f'{e[0]}_{e[1].upper()}' for e in agg.columns])
    agg['BURO_LOAN_COUNT'] = bureau.groupby('SK_ID_CURR').size()

    del bureau; gc.collect()
    return agg


def _previous_app_aggregates(data_dir: str) -> pd.DataFrame:
    prev = pd.read_csv(f'{data_dir}/previous_application.csv')
    for c in ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE',
              'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE',
              'DAYS_TERMINATION']:
        prev[c].replace(365243, np.nan, inplace=True)

    prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
    prev.replace([np.inf, -np.inf], np.nan, inplace=True)

    aggs = {
        'AMT_ANNUITY':       ['mean'],
        'AMT_CREDIT':        ['mean', 'sum'],
        'AMT_APPLICATION':   ['mean'],
        'APP_CREDIT_PERC':   ['mean'],
        'AMT_DOWN_PAYMENT':  ['mean'],
        'CNT_PAYMENT':       ['mean'],
        'DAYS_DECISION':     ['mean', 'max'],
        'RATE_DOWN_PAYMENT': ['mean'],
    }
    agg = prev.groupby('SK_ID_CURR').agg(aggs)
    agg.columns = pd.Index([f'{e[0]}_{e[1].upper()}' for e in agg.columns])
    agg['PREV_APP_COUNT'] = prev.groupby('SK_ID_CURR').size()

    del prev; gc.collect()
    return agg


# ============================================================================
# 2. CLUSTERING WRAPPER (FIXED)
# ============================================================================
def _fit_clusters(features: pd.DataFrame,
                  n_clusters: int,
                  method: str = 'kmeans',
                  prefix: str = 'CLUSTER',
                  seed: int = 42
                  ) -> pd.DataFrame:
    """
    Fit clustering with robust preprocessing.

    Changes from previous version:
      - log-transform skewed columns
      - clip outliers at 99 percentile
      - default to KMeans (more robust on skewed data)
    """
    X, _, _ = _preprocess_for_clustering(features, clip_pct=0.99)

    if method == 'gmm':
        model = GaussianMixture(n_components=n_clusters,
                                random_state=seed,
                                n_init=5,                    # ↑ from 3
                                covariance_type='diag',      # ← changed from 'full'
                                reg_covar=1e-3,              # ← prevent singular cov
                                max_iter=200)
        model.fit(X)
        assignment = model.predict(X)
        scores = model.predict_proba(X)
        score_cols = [f'{prefix}_PROB_{i}' for i in range(n_clusters)]
    elif method == 'kmeans':
        model = KMeans(n_clusters=n_clusters,
                       random_state=seed,
                       n_init=10,
                       max_iter=300)
        model.fit(X)
        assignment = model.predict(X)
        scores = model.transform(X)
        score_cols = [f'{prefix}_DIST_{i}' for i in range(n_clusters)]
    else:
        raise ValueError(method)

    # Verify cluster balance
    unique, counts = np.unique(assignment, return_counts=True)
    proportions = counts / len(assignment)
    print(f'    Cluster sizes: {[f"{p:.1%}" for p in proportions]}')

    if proportions.max() > 0.7:
        print(f'    WARNING: dominant cluster {proportions.max():.1%} - clustering may be degenerate')

    out = pd.DataFrame(scores, index=features.index, columns=score_cols)
    out[f'{prefix}_LABEL'] = assignment.astype(np.int8)
    out[score_cols] = out[score_cols].astype(np.float32)

    return out


# ============================================================================
# MAIN
# ============================================================================
def build_clustering_features(data_dir: str,
                              n_clusters_bureau: int = 4,    # ↓ from 5
                              n_clusters_prev: int = 5,
                              seed: int = 42
                              ) -> pd.DataFrame:
    """
    Build clustering features.

    Changes:
      - n_clusters_bureau reduced to 4 (5 was too many for the data shape)
      - Both use KMeans (GMM was collapsing)
      - Proper log-transform + outlier clipping in preprocessing
    """
    print('=' * 60)
    print('Clustering Features (v2 - fixed)')
    print('=' * 60)

    with timer('bureau aggregates'):
        bur_agg = _bureau_aggregates(data_dir)
        print(f'  shape: {bur_agg.shape}')

    with timer('previous_app aggregates'):
        prev_agg = _previous_app_aggregates(data_dir)
        print(f'  shape: {prev_agg.shape}')

    with timer(f'KMeans bureau ({n_clusters_bureau} clusters)'):
        bur_cluster = _fit_clusters(bur_agg, n_clusters_bureau,
                                     method='kmeans', prefix='BURO_CLU', seed=seed)
        print(f'  shape: {bur_cluster.shape}')

    with timer(f'KMeans previous_app ({n_clusters_prev} clusters)'):
        prev_cluster = _fit_clusters(prev_agg, n_clusters_prev,
                                      method='kmeans', prefix='PREV_CLU', seed=seed)
        print(f'  shape: {prev_cluster.shape}')

    cluster_df = bur_cluster.join(prev_cluster, how='outer')

    print(f'\nFinal clustering features: {cluster_df.shape}')
    print(f'Memory: {cluster_df.memory_usage(deep=True).sum()/1024**2:.1f} MB')

    return cluster_df


if __name__ == '__main__':
    DATA_DIR = 'data/raw'
    clu = build_clustering_features(DATA_DIR)
    clu.to_parquet('./clustering_features.parquet')   # OVERWRITE old version
    print('Saved -> ./clustering_features.parquet')

Clustering Features (v2 - fixed)
[bureau aggregates] start
  shape: (305811, 15)
[bureau aggregates] done in 4.7s
[previous_app aggregates] start
  shape: (338857, 11)
[previous_app aggregates] done in 12.7s
[KMeans bureau (4 clusters)] start
    Cluster sizes: ['13.3%', '41.8%', '16.2%', '28.7%']
  shape: (305811, 5)
[KMeans bureau (4 clusters)] done in 3.8s
[KMeans previous_app (5 clusters)] start
    Cluster sizes: ['20.7%', '8.8%', '28.4%', '18.5%', '23.5%']
  shape: (338857, 6)
[KMeans previous_app (5 clusters)] done in 4.0s

Final clustering features: (353577, 11)
Memory: 20.2 MB
Saved -> ./clustering_features.parquet


In [ ]:
import pandas as pd
clu = pd.read_parquet('./clustering_features.parquet')

# 1. Cluster distribution — expect roughly balanced
print('=== BURO_CLU_LABEL distribution ===')
print(clu['BURO_CLU_LABEL'].value_counts(normalize=True).sort_index().round(3))

print('\n=== PREV_CLU_LABEL distribution ===')
print(clu['PREV_CLU_LABEL'].value_counts(normalize=True).sort_index().round(3))

# 2. Probability/distance ranges
print('\n=== Stats ===')
print(clu.describe().T[['mean', 'std', 'min', 'max']].round(3))

=== BURO_CLU_LABEL distribution ===
BURO_CLU_LABEL
0.0    0.016
1.0    0.153
2.0    0.010
3.0    0.822
4.0    0.000
Name: proportion, dtype: float64

=== PREV_CLU_LABEL distribution ===
PREV_CLU_LABEL
0.0    0.075
1.0    0.099
2.0    0.515
3.0    0.058
4.0    0.253
Name: proportion, dtype: float64

=== Stats ===
                  mean    std    min      max
BURO_CLU_PROB_0  0.016  0.124  0.000    1.000
BURO_CLU_PROB_1  0.154  0.355  0.000    1.000
BURO_CLU_PROB_2  0.010  0.096  0.000    1.000
BURO_CLU_PROB_3  0.821  0.377  0.000    1.000
BURO_CLU_PROB_4  0.000  0.004  0.000    1.000
BURO_CLU_LABEL   2.637  0.795  0.000    4.000
PREV_CLU_DIST_0  4.382  1.656  0.653  114.192
PREV_CLU_DIST_1  4.373  1.850  0.214  115.898
PREV_CLU_DIST_2  2.833  2.040  0.261  116.032
PREV_CLU_DIST_3  6.196  1.629  0.677  114.922
PREV_CLU_DIST_4  3.284  1.652  0.416  115.705
PREV_CLU_LABEL   2.316  1.171  0.000    4.000


In [ ]:
"""merge_extra_features.py"""
import pandas as pd
import numpy as np

# Load existing
train = pd.read_parquet('./train_merged.parquet')
test  = pd.read_parquet('./test_merged.parquet')
print(f'Before: train={train.shape}, test={test.shape}')

# Load new
trend = pd.read_parquet('./trend_features.parquet')
clu   = pd.read_parquet('./clustering_features.parquet')
print(f'Trend: {trend.shape}, Clustering: {clu.shape}')

# Set index name and merge
trend.index.name = 'SK_ID_CURR'
clu.index.name   = 'SK_ID_CURR'

train = train.merge(trend.reset_index(), on='SK_ID_CURR', how='left')
train = train.merge(clu.reset_index(),   on='SK_ID_CURR', how='left')

test = test.merge(trend.reset_index(), on='SK_ID_CURR', how='left')
test = test.merge(clu.reset_index(),   on='SK_ID_CURR', how='left')

print(f'\nAfter: train={train.shape}, test={test.shape}')
print(f'Expected: train=(307511, {409 + 40 + 11}), test=(48744, {408 + 40 + 11})')

# Save with v2 tag (KHÔNG overwrite original)
train.to_parquet('./train_merged_v2.parquet')
test.to_parquet('./test_merged_v2.parquet')
print('\nSaved -> ./train_merged_v2.parquet, test_merged_v2.parquet')

Before: train=(307511, 409), test=(48744, 408)
Trend: (343907, 40), Clustering: (353577, 11)

After: train=(307511, 460), test=(48744, 459)
Expected: train=(307511, 460), test=(48744, 459)

Saved -> ./train_merged_v2.parquet, test_merged_v2.parquet


### Feature selection / cleanup

This step reduces redundant/noisy predictors and prepares cleaner feature versions for downstream models.


### Feature Selection

In [ ]:
"""
feature_selection.py
Drop low-importance features từ feature importance đã save.
"""
import pandas as pd
import numpy as np

# Load importance từ training
fi = pd.read_csv('./output/feature_importance_lgb_v3_with_trend_cluster.csv')
print(f'Total features: {len(fi)}')

# Strategy: drop bottom 25% by importance
# Why 25%? Empirical sweet spot - keeps high-mid signal, drops pure noise
threshold = fi['importance'].quantile(0.25)
keep_features = fi[fi['importance'] > threshold]['feature'].tolist()
drop_features = fi[fi['importance'] <= threshold]['feature'].tolist()

print(f'Keeping: {len(keep_features)} features')
print(f'Dropping: {len(drop_features)} features')
print(f'\nLowest 10 features being dropped:')
print(fi.nsmallest(10, 'importance').to_string(index=False))

# Apply to train/test
train = pd.read_parquet('./train_merged_v2.parquet')
test  = pd.read_parquet('./test_merged_v2.parquet')

# Always keep TARGET + SK_ID_CURR
keep_in_train = ['TARGET', 'SK_ID_CURR'] + [c for c in keep_features if c in train.columns]
keep_in_test  = ['SK_ID_CURR'] + [c for c in keep_features if c in test.columns]

train_v3 = train[keep_in_train]
test_v3  = test[keep_in_test]

print(f'\nNew shape: train={train_v3.shape}, test={test_v3.shape}')

train_v3.to_parquet('./train_merged_v3.parquet')
test_v3.to_parquet('./test_merged_v3.parquet')
print('Saved -> train_merged_v3.parquet, test_merged_v3.parquet')

Total features: 458
Keeping: 343 features
Dropping: 115 features

Lowest 10 features being dropped:
                                        feature  importance
         bur_Loan_for_the_purchase_of_equipment         0.0
                   bur_Cash_loan_non_earmarked_         0.0
bur_Loan_for_purchase_of_shares_margin_lending_         0.0
                                 bur_currency_4         0.0
                                 bur_currency_3         0.0
                                 bur_currency_2         0.0
                               bur_sold_count_x         0.0
                                  ccb_count_rej         0.0
                                  bur_bad_count         0.0
                            pcb_end_as_Canceled         0.0

New shape: train=(307511, 313), test=(48744, 312)
Saved -> train_merged_v3.parquet, test_merged_v3.parquet


### Deep behavioral aggregations

Bureau deep features capture external credit history. Previous-application features capture approval/refusal and historical application behavior. Installment features capture late payment, underpayment, and repayment behavior. Cross-table features compare external debt/history with the current application.

Heavy cells: do not run during demo unless local artifacts and sufficient runtime are available.


### **LightBGM v2**

In [ ]:
# ========== CELL FE2-1: SETUP ==========
import numpy as np
import pandas as pd
import gc
import time
from pathlib import Path
from contextlib import contextmanager

@contextmanager
def timer(name):
    t = time.time()
    yield
    print(f'[{name}] {time.time()-t:.1f}s')

# Load raw tables — bạn đã có sẵn từ training trước
DATA_DIR = Path('data/raw')

with timer('Load raw tables'):
    bureau = pd.read_csv(DATA_DIR / 'bureau.csv')
    bureau_balance = pd.read_csv(DATA_DIR / 'bureau_balance.csv')
    previous_application = pd.read_csv(DATA_DIR / 'previous_application.csv')
    pos_cash = pd.read_csv(DATA_DIR / 'POS_CASH_balance.csv')
    installments = pd.read_csv(DATA_DIR / 'installments_payments.csv')
    credit_card = pd.read_csv(DATA_DIR / 'credit_card_balance.csv')
    application_train = pd.read_csv(DATA_DIR / 'application_train.csv')
    application_test = pd.read_csv(DATA_DIR / 'application_test.csv')

print(f'Bureau:               {bureau.shape}')
print(f'Bureau balance:       {bureau_balance.shape}')
print(f'Previous application: {previous_application.shape}')
print(f'POS cash:             {pos_cash.shape}')
print(f'Installments:         {installments.shape}')
print(f'Credit card:          {credit_card.shape}')
print(f'Application train:    {application_train.shape}')
print(f'Application test:     {application_test.shape}')

[Load raw tables] 45.0s
Bureau:               (1716428, 17)
Bureau balance:       (27299925, 3)
Previous application: (1670214, 37)
POS cash:             (10001358, 8)
Installments:         (13605401, 8)
Credit card:          (3840312, 23)
Application train:    (307511, 122)
Application test:     (48744, 121)


In [ ]:
# ========== CELL FE2-2: BUREAU DEEP ==========
with timer('Bureau deep aggregations'):
    bur = bureau.copy()

    # === 1. Per credit type aggregations ===
    # Get top 6 credit types only
    top_types = bur['CREDIT_TYPE'].value_counts().head(6).index.tolist()
    bur['CREDIT_TYPE_GROUPED'] = bur['CREDIT_TYPE'].where(
        bur['CREDIT_TYPE'].isin(top_types), 'Other'
    )

    bureau_by_type = bur.groupby(['SK_ID_CURR', 'CREDIT_TYPE_GROUPED']).agg({
        'SK_ID_BUREAU': 'count',
        'DAYS_CREDIT': ['mean', 'max', 'min'],
        'AMT_CREDIT_SUM': ['mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['mean', 'sum'],
        'CREDIT_DAY_OVERDUE': 'max',
    }).reset_index()

    # Flatten columns
    bureau_by_type.columns = ['SK_ID_CURR', 'TYPE'] + [
        f'BUR_TYPE_{a}_{b}' for a, b in bureau_by_type.columns[2:]
    ]

    # Pivot to wide format
    bureau_type_wide = bureau_by_type.pivot_table(
        index='SK_ID_CURR',
        columns='TYPE',
        aggfunc='first',
    )
    bureau_type_wide.columns = [
        f'{col[0]}_{col[1]}' for col in bureau_type_wide.columns
    ]
    bureau_type_wide = bureau_type_wide.reset_index()

    # === 2. Active vs Closed ratios ===
    active = bur[bur['CREDIT_ACTIVE'] == 'Active']
    closed = bur[bur['CREDIT_ACTIVE'] == 'Closed']

    active_agg = active.groupby('SK_ID_CURR').agg({
        'SK_ID_BUREAU': 'count',
        'AMT_CREDIT_SUM': 'sum',
        'AMT_CREDIT_SUM_DEBT': 'sum',
        'CREDIT_DAY_OVERDUE': 'sum',
    }).rename(columns=lambda x: f'BUR_ACTIVE_{x}')

    closed_agg = closed.groupby('SK_ID_CURR').agg({
        'SK_ID_BUREAU': 'count',
        'AMT_CREDIT_SUM': 'sum',
        'AMT_CREDIT_SUM_DEBT': 'sum',
    }).rename(columns=lambda x: f'BUR_CLOSED_{x}')

    # === 3. Recent (last year) vs Old performance ===
    recent_bur = bur[bur['DAYS_CREDIT'] > -365]
    old_bur = bur[bur['DAYS_CREDIT'] <= -365]

    recent_agg = recent_bur.groupby('SK_ID_CURR').agg({
        'SK_ID_BUREAU': 'count',
        'AMT_CREDIT_SUM': ['sum', 'mean'],
        'CREDIT_DAY_OVERDUE': 'max',
    })
    recent_agg.columns = [f'BUR_RECENT_{a}_{b}' for a, b in recent_agg.columns]

    old_agg = old_bur.groupby('SK_ID_CURR').agg({
        'SK_ID_BUREAU': 'count',
        'AMT_CREDIT_SUM': ['sum', 'mean'],
        'CREDIT_DAY_OVERDUE': 'max',
    })
    old_agg.columns = [f'BUR_OLD_{a}_{b}' for a, b in old_agg.columns]

    # === 4. Combine all ===
    bureau_deep = bureau_type_wide.set_index('SK_ID_CURR')
    bureau_deep = bureau_deep.join(active_agg, how='outer')
    bureau_deep = bureau_deep.join(closed_agg, how='outer')
    bureau_deep = bureau_deep.join(recent_agg, how='outer')
    bureau_deep = bureau_deep.join(old_agg, how='outer')

    # === 5. Compute ratios ===
    bureau_deep['BUR_ACTIVE_RATIO'] = (
        bureau_deep['BUR_ACTIVE_SK_ID_BUREAU'] /
        (bureau_deep['BUR_ACTIVE_SK_ID_BUREAU'].fillna(0) +
         bureau_deep['BUR_CLOSED_SK_ID_BUREAU'].fillna(0) + 1)
    )
    bureau_deep['BUR_RECENT_VS_OLD_COUNT_RATIO'] = (
        bureau_deep['BUR_RECENT_SK_ID_BUREAU_count'] /
        (bureau_deep['BUR_OLD_SK_ID_BUREAU_count'].fillna(0) + 1)
    )
    bureau_deep['BUR_RECENT_VS_OLD_AMT_RATIO'] = (
        bureau_deep['BUR_RECENT_AMT_CREDIT_SUM_sum'] /
        (bureau_deep['BUR_OLD_AMT_CREDIT_SUM_sum'].fillna(0) + 1)
    )

    bureau_deep = bureau_deep.reset_index()

print(f'Bureau deep features: {bureau_deep.shape}')
print(f'Sample columns:')
for c in bureau_deep.columns[:10]:
    print(f'  {c}')

[Bureau deep aggregations] 3.9s
Bureau deep features: (305811, 82)
Sample columns:
  SK_ID_CURR
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_mean_Car loan
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_mean_Consumer credit
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_mean_Credit card
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_mean_Loan for business development
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_mean_Microloan
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_mean_Mortgage
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_mean_Other
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_sum_Car loan
  BUR_TYPE_AMT_CREDIT_SUM_DEBT_sum_Consumer credit


In [ ]:
# ========== CELL FE2-3: PREVIOUS APPLICATION DETAILED ==========
with timer('Previous application detailed'):
    prev = previous_application.copy()

    # === 1. Approved vs Refused breakdowns ===
    approved = prev[prev['NAME_CONTRACT_STATUS'] == 'Approved']
    refused = prev[prev['NAME_CONTRACT_STATUS'] == 'Refused']
    canceled = prev[prev['NAME_CONTRACT_STATUS'] == 'Canceled']

    approved_agg = approved.groupby('SK_ID_CURR').agg({
        'SK_ID_PREV': 'count',
        'AMT_CREDIT': ['sum', 'mean', 'max'],
        'AMT_ANNUITY': ['sum', 'mean'],
        'CNT_PAYMENT': ['mean', 'sum'],
        'DAYS_DECISION': ['max', 'min'],
    })
    approved_agg.columns = [f'PREV_APPROVED_{a}_{b}' for a, b in approved_agg.columns]

    refused_agg = refused.groupby('SK_ID_CURR').agg({
        'SK_ID_PREV': 'count',
        'AMT_APPLICATION': ['sum', 'mean'],
        'DAYS_DECISION': 'max',
    })
    refused_agg.columns = [f'PREV_REFUSED_{a}_{b}' for a, b in refused_agg.columns]

    canceled_agg = canceled.groupby('SK_ID_CURR').agg({
        'SK_ID_PREV': 'count',
    })
    canceled_agg.columns = ['PREV_CANCELED_count']

    # === 2. Total previous applications ===
    total_agg = prev.groupby('SK_ID_CURR').agg({
        'SK_ID_PREV': 'count',
        'DAYS_DECISION': ['max', 'min'],
    })
    total_agg.columns = [f'PREV_TOTAL_{a}_{b}' for a, b in total_agg.columns]

    # === 3. Combine ===
    prev_detailed = total_agg.join(approved_agg, how='outer')
    prev_detailed = prev_detailed.join(refused_agg, how='outer')
    prev_detailed = prev_detailed.join(canceled_agg, how='outer')

    # === 4. Compute ratios ===
    total_count = prev_detailed['PREV_TOTAL_SK_ID_PREV_count'].fillna(0) + 1

    prev_detailed['PREV_REFUSED_RATE'] = (
        prev_detailed['PREV_REFUSED_SK_ID_PREV_count'].fillna(0) / total_count
    )
    prev_detailed['PREV_APPROVED_RATE'] = (
        prev_detailed['PREV_APPROVED_SK_ID_PREV_count'].fillna(0) / total_count
    )
    prev_detailed['PREV_CANCELED_RATE'] = (
        prev_detailed['PREV_CANCELED_count'].fillna(0) / total_count
    )

    # Days since last application
    prev_detailed['PREV_DAYS_SINCE_LAST'] = -prev_detailed['PREV_TOTAL_DAYS_DECISION_max']

    # Application velocity (apps per year of history)
    history_span = (
        prev_detailed['PREV_TOTAL_DAYS_DECISION_max'] -
        prev_detailed['PREV_TOTAL_DAYS_DECISION_min']
    ).abs() / 365 + 1
    prev_detailed['PREV_APP_VELOCITY'] = (
        prev_detailed['PREV_TOTAL_SK_ID_PREV_count'] / history_span
    )

    prev_detailed = prev_detailed.reset_index()

print(f'Previous application detailed: {prev_detailed.shape}')

[Previous application detailed] 2.1s
Previous application detailed: (338857, 24)


In [ ]:
# ========== CELL FE2-4: INSTALLMENT BEHAVIOR ==========
with timer('Installment behavior'):
    inst = installments.copy()

    # Compute payment behavior fields
    inst['LATE_DAYS'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
    inst['IS_LATE'] = (inst['LATE_DAYS'] > 0).astype(int)
    inst['IS_EARLY'] = (inst['LATE_DAYS'] < 0).astype(int)
    inst['PAYMENT_DIFF'] = inst['AMT_PAYMENT'] - inst['AMT_INSTALMENT']
    inst['IS_UNDERPAY'] = (inst['PAYMENT_DIFF'] < 0).astype(int)
    inst['IS_OVERPAY'] = (inst['PAYMENT_DIFF'] > 0).astype(int)

    # === 1. Overall behavior ===
    inst_overall = inst.groupby('SK_ID_CURR').agg({
        'IS_LATE': ['sum', 'mean'],
        'IS_EARLY': ['sum', 'mean'],
        'IS_UNDERPAY': ['sum', 'mean'],
        'IS_OVERPAY': ['sum', 'mean'],
        'LATE_DAYS': ['mean', 'max', 'std'],
        'PAYMENT_DIFF': ['mean', 'sum', 'std'],
    })
    inst_overall.columns = [f'INST_OVERALL_{a}_{b}' for a, b in inst_overall.columns]

    # === 2. Recent (last 365 days) ===
    recent_inst = inst[inst['DAYS_INSTALMENT'] > -365]
    inst_recent = recent_inst.groupby('SK_ID_CURR').agg({
        'IS_LATE': ['sum', 'mean'],
        'IS_UNDERPAY': ['sum', 'mean'],
        'LATE_DAYS': ['mean', 'max'],
        'NUM_INSTALMENT_NUMBER': 'count',
    })
    inst_recent.columns = [f'INST_RECENT_{a}_{b}' for a, b in inst_recent.columns]

    # === 3. Last 90 days ===
    last90_inst = inst[inst['DAYS_INSTALMENT'] > -90]
    inst_90 = last90_inst.groupby('SK_ID_CURR').agg({
        'IS_LATE': ['sum', 'mean'],
        'LATE_DAYS': 'max',
    })
    inst_90.columns = [f'INST_LAST90_{a}_{b}' for a, b in inst_90.columns]

    # Combine
    inst_behavior = inst_overall.join(inst_recent, how='outer')
    inst_behavior = inst_behavior.join(inst_90, how='outer')

    # Worsening trend signal
    inst_behavior['INST_LATE_TREND'] = (
        inst_behavior['INST_RECENT_IS_LATE_mean'].fillna(0) -
        inst_behavior['INST_OVERALL_IS_LATE_mean'].fillna(0)
    )

    inst_behavior = inst_behavior.reset_index()

print(f'Installment behavior: {inst_behavior.shape}')

[Installment behavior] 6.0s
Installment behavior: (339587, 26)


In [ ]:
# ========== CELL FE2-5: CROSS-TABLE FEATURES ==========
with timer('Cross-table features'):
    # Need application data
    app = application_train[['SK_ID_CURR', 'AMT_CREDIT', 'AMT_INCOME_TOTAL',
                              'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']].copy()
    app_test = application_test[['SK_ID_CURR', 'AMT_CREDIT', 'AMT_INCOME_TOTAL',
                                  'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']].copy()
    app_combined = pd.concat([app, app_test], ignore_index=True)

    # === 1. Bureau active debt aggregation ===
    bur_active_debt = bureau[bureau['CREDIT_ACTIVE'] == 'Active'].groupby('SK_ID_CURR').agg({
        'AMT_CREDIT_SUM_DEBT': 'sum',
        'AMT_CREDIT_SUM': 'sum',
        'DAYS_CREDIT': 'min',
    }).rename(columns={
        'AMT_CREDIT_SUM_DEBT': 'BUR_TOTAL_ACTIVE_DEBT',
        'AMT_CREDIT_SUM': 'BUR_TOTAL_ACTIVE_AMT',
        'DAYS_CREDIT': 'BUR_OLDEST_CREDIT_DAY',
    })

    # === 2. Previous open application amount ===
    prev_open = previous_application[
        previous_application['NAME_CONTRACT_STATUS'].isin(['Approved'])
    ].groupby('SK_ID_CURR').agg({
        'AMT_CREDIT': 'sum',
    }).rename(columns={'AMT_CREDIT': 'PREV_TOTAL_APPROVED_AMT'})

    # === 3. Build cross-table df ===
    cross = app_combined.set_index('SK_ID_CURR')
    cross = cross.join(bur_active_debt, how='left')
    cross = cross.join(prev_open, how='left')

    # Fill NaN với 0 (no history)
    cross['BUR_TOTAL_ACTIVE_DEBT'] = cross['BUR_TOTAL_ACTIVE_DEBT'].fillna(0)
    cross['BUR_TOTAL_ACTIVE_AMT'] = cross['BUR_TOTAL_ACTIVE_AMT'].fillna(0)
    cross['PREV_TOTAL_APPROVED_AMT'] = cross['PREV_TOTAL_APPROVED_AMT'].fillna(0)

    # === 4. Compute cross-table ratios ===
    # Total debt across all sources
    cross['CROSS_TOTAL_DEBT'] = (
        cross['AMT_CREDIT'] +
        cross['BUR_TOTAL_ACTIVE_DEBT'] +
        cross['PREV_TOTAL_APPROVED_AMT']
    )

    # Debt to income ratios
    cross['CROSS_TOTAL_DEBT_TO_INCOME'] = (
        cross['CROSS_TOTAL_DEBT'] / (cross['AMT_INCOME_TOTAL'] + 1)
    )
    cross['CROSS_BUR_DEBT_TO_INCOME'] = (
        cross['BUR_TOTAL_ACTIVE_DEBT'] / (cross['AMT_INCOME_TOTAL'] + 1)
    )
    cross['CROSS_CURRENT_TO_TOTAL_DEBT'] = (
        cross['AMT_CREDIT'] / (cross['CROSS_TOTAL_DEBT'] + 1)
    )

    # Annuity burden
    cross['CROSS_ANNUITY_TO_INCOME'] = (
        cross['AMT_ANNUITY'] / (cross['AMT_INCOME_TOTAL'] + 1)
    )

    # Credit history depth (years)
    cross['CROSS_HISTORY_DEPTH_YEARS'] = (
        cross['BUR_OLDEST_CREDIT_DAY'].abs() / 365
    ).fillna(0)

    # History depth relative to age
    cross['CROSS_HISTORY_TO_AGE_RATIO'] = (
        cross['CROSS_HISTORY_DEPTH_YEARS'] * 365 /
        cross['DAYS_BIRTH'].abs()
    ).fillna(0)

    # Debt utilization signal
    cross['CROSS_DEBT_UTILIZATION'] = (
        cross['BUR_TOTAL_ACTIVE_DEBT'] /
        (cross['BUR_TOTAL_ACTIVE_AMT'] + 1)
    )

    # Keep only new features (drop the merged inputs)
    cross_features = cross[[
        'BUR_TOTAL_ACTIVE_DEBT', 'BUR_TOTAL_ACTIVE_AMT',
        'PREV_TOTAL_APPROVED_AMT', 'CROSS_TOTAL_DEBT',
        'CROSS_TOTAL_DEBT_TO_INCOME', 'CROSS_BUR_DEBT_TO_INCOME',
        'CROSS_CURRENT_TO_TOTAL_DEBT', 'CROSS_ANNUITY_TO_INCOME',
        'CROSS_HISTORY_DEPTH_YEARS', 'CROSS_HISTORY_TO_AGE_RATIO',
        'CROSS_DEBT_UTILIZATION',
    ]].reset_index()

print(f'Cross-table features: {cross_features.shape}')
print(f'New cross features:')
for c in cross_features.columns[1:]:
    print(f'  {c}')

[Cross-table features] 1.1s
Cross-table features: (356255, 12)
New cross features:
  BUR_TOTAL_ACTIVE_DEBT
  BUR_TOTAL_ACTIVE_AMT
  PREV_TOTAL_APPROVED_AMT
  CROSS_TOTAL_DEBT
  CROSS_TOTAL_DEBT_TO_INCOME
  CROSS_BUR_DEBT_TO_INCOME
  CROSS_CURRENT_TO_TOTAL_DEBT
  CROSS_ANNUITY_TO_INCOME
  CROSS_HISTORY_DEPTH_YEARS
  CROSS_HISTORY_TO_AGE_RATIO
  CROSS_DEBT_UTILIZATION


In [ ]:
# ========== CELL FE2-6: SAVE FE V2 ==========
FE_V2_DIR = Path('./fe_v2')
FE_V2_DIR.mkdir(exist_ok=True)

bureau_deep.to_parquet(FE_V2_DIR / 'bureau_deep.parquet', index=False)
prev_detailed.to_parquet(FE_V2_DIR / 'prev_detailed.parquet', index=False)
inst_behavior.to_parquet(FE_V2_DIR / 'inst_behavior.parquet', index=False)
cross_features.to_parquet(FE_V2_DIR / 'cross_features.parquet', index=False)

print(f'Saved FE V2 features:')
print(f'  bureau_deep:     {bureau_deep.shape}')
print(f'  prev_detailed:   {prev_detailed.shape}')
print(f'  inst_behavior:   {inst_behavior.shape}')
print(f'  cross_features:  {cross_features.shape}')
print(f'  Total new features: {(bureau_deep.shape[1]-1) + (prev_detailed.shape[1]-1) + (inst_behavior.shape[1]-1) + (cross_features.shape[1]-1)}')

# Clean memory
del bureau, bureau_balance, previous_application, pos_cash, installments, credit_card
del application_train, application_test, app, app_test, app_combined
del bur, prev, inst, recent_bur, old_bur, approved, refused, canceled
gc.collect()

Saved FE V2 features:
  bureau_deep:     (305811, 82)
  prev_detailed:   (338857, 24)
  inst_behavior:   (339587, 26)
  cross_features:  (356255, 12)
  Total new features: 140


0

In [ ]:
# ========== CELL FE2-7: MERGE TO V7 ==========
with timer('Load v6 and merge'):
    train_v7 = pd.read_parquet('train_merged_v6.parquet')
    test_v7 = pd.read_parquet('test_merged_v6.parquet')

    print(f'Before merge:')
    print(f'  Train: {train_v7.shape}')
    print(f'  Test:  {test_v7.shape}')

with timer('Merge FE V2 features'):
    fe_v2_dfs = [bureau_deep, prev_detailed, inst_behavior, cross_features]

    for df_new in fe_v2_dfs:
        # Drop columns that already exist in train_v7 (avoid duplicates)
        new_cols = [c for c in df_new.columns
                    if c == 'SK_ID_CURR' or c not in train_v7.columns]
        df_filtered = df_new[new_cols]

        train_v7 = train_v7.merge(df_filtered, on='SK_ID_CURR', how='left')
        test_v7 = test_v7.merge(df_filtered, on='SK_ID_CURR', how='left')

# Reduce memory
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and col_type.name != 'category':
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.api.types.is_integer_dtype(col_type):
                if c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

with timer('Reduce memory'):
    train_v7 = reduce_mem_usage(train_v7)
    test_v7 = reduce_mem_usage(test_v7)

print(f'\nAfter merge V7:')
print(f'  Train: {train_v7.shape}')
print(f'  Test:  {test_v7.shape}')
print(f'  New features added: {train_v7.shape[1] - 501}')
print(f'  Memory train: {train_v7.memory_usage().sum() / 1e9:.2f} GB')

# Save
train_v7.to_parquet('train_merged_v7.parquet', index=False)
test_v7.to_parquet('test_merged_v7.parquet', index=False)
print(f'\nSaved train_merged_v7.parquet and test_merged_v7.parquet')

del fe_v2_dfs
gc.collect()

Before merge:
  Train: (307507, 501)
  Test:  (48744, 500)
[Load v6 and merge] 1.4s
[Merge FE V2 features] 6.5s
[Reduce memory] 1.7s

After merge V7:
  Train: (307507, 641)
  Test:  (48744, 640)
  New features added: 140
  Memory train: 0.81 GB

Saved train_merged_v7.parquet and test_merged_v7.parquet


13897

### Model-based auxiliary features

These features use auxiliary prediction tasks to enrich missing or latent risk-related signals. They should be used carefully to avoid leakage; the target of the main credit-default task is not directly leaked into validation folds.

Heavy cells: do not run during demo unless local artifacts and sufficient runtime are available.


### **TARGET NN500 Feature**

#### Add predicted features

### **Predicted Interest Rate**

In [ ]:
# ========== PHASE X1.1: COMPUTE IR FROM PREV_APP ==========
import numpy as np
import pandas as pd

prev_app = pd.read_csv('data/raw/previous_application.csv')
print(f'previous_application: {prev_app.shape}')

def compute_ir(credit, annuity, n_payments):
    """Approximate annual interest rate."""
    valid = (credit > 0) & (annuity > 0) & (n_payments > 0)
    ir = np.full(len(credit), np.nan, dtype=np.float32)

    total_paid = annuity * n_payments
    total_interest = total_paid - credit
    ir_approx = total_interest / (credit * n_payments / 12.0 + 1)
    ir[valid] = ir_approx[valid]
    return ir

prev_app['IR_COMPUTED'] = compute_ir(
    prev_app['AMT_CREDIT'].values,
    prev_app['AMT_ANNUITY'].values,
    prev_app['CNT_PAYMENT'].values,
)
prev_app['IR_COMPUTED'] = prev_app['IR_COMPUTED'].clip(0, 5)

print(f'IR stats:')
print(prev_app['IR_COMPUTED'].describe())
print(f'Non-NaN: {prev_app["IR_COMPUTED"].notna().sum()} / {len(prev_app)}')

previous_application: (1670214, 37)
IR stats:
count    1.152941e+06
mean     2.588013e-01
std      1.224515e-01
min      0.000000e+00
25%      1.520954e-01
50%      2.324693e-01
75%      3.570760e-01
max      8.227639e-01
Name: IR_COMPUTED, dtype: float64
Non-NaN: 1152941 / 1670214


In [ ]:
# ========== PHASE X1.2: TRAIN IR PREDICTOR ==========
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
import gc, time

ir_train_data = prev_app[prev_app['IR_COMPUTED'].notna()].copy()
print(f'IR training samples: {len(ir_train_data)}')

COMMON_FEATURES = [
    'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CNT_PAYMENT', 'NAME_CONTRACT_TYPE',
    'NAME_PAYMENT_TYPE', 'NAME_GOODS_CATEGORY',
    'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE',
    'CHANNEL_TYPE', 'NAME_YIELD_GROUP',
    'PRODUCT_COMBINATION',
]
prev_features = [f for f in COMMON_FEATURES if f in ir_train_data.columns]
print(f'Features: {prev_features}')

X_ir = ir_train_data[prev_features].copy()
y_ir = ir_train_data['IR_COMPUTED'].values

cat_cols_ir = []
for col in prev_features:
    if X_ir[col].dtype == 'object':
        X_ir[col] = X_ir[col].fillna('MISSING').astype(str)
        codes, _ = pd.factorize(X_ir[col])
        X_ir[col] = codes.astype(np.int32)
        cat_cols_ir.append(col)

X_ir = X_ir.replace([np.inf, -np.inf], np.nan)

LGB_IR_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6,
    'min_child_samples': 100, 'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'colsample_bytree': 0.8, 'subsample': 0.8, 'subsample_freq': 1,
    'verbosity': -1, 'random_state': 42, 'n_jobs': -1,
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ir_models = []
rmse_scores = []

print('Training IR predictor...')
print('=' * 70)
start = time.time()

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_ir)):
    X_tr, X_val = X_ir.iloc[tr_idx], X_ir.iloc[val_idx]
    y_tr, y_val = y_ir[tr_idx], y_ir[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols_ir)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols_ir)

    model = lgb.train(
        LGB_IR_PARAMS, dtrain, num_boost_round=2000,
        valid_sets=[dval], valid_names=['valid'],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    r2 = r2_score(y_val, val_preds)

    rmse_scores.append(rmse)
    ir_models.append(model)

    print(f'Fold {fold_idx+1}/5 | RMSE: {rmse:.4f} | R²: {r2:.4f} | best_iter: {model.best_iteration}')
    del dtrain, dval
    gc.collect()

print('=' * 70)
print(f'Mean RMSE: {np.mean(rmse_scores):.4f}')
print(f'Time: {time.time()-start:.0f}s')

IR training samples: 1152941
Features: ['AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CNT_PAYMENT', 'NAME_CONTRACT_TYPE', 'NAME_PAYMENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION']
Training IR predictor...
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid's rmse: 0.0222475
Fold 1/5 | RMSE: 0.0222 | R²: 0.9671 | best_iter: 2000
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid's rmse: 0.0223221
Fold 2/5 | RMSE: 0.0223 | R²: 0.9668 | best_iter: 2000
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid's rmse: 0.0223447
Fold 3/5 | RMSE: 0.0223 | R²: 0.9667 | best_iter: 2000
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	val

In [ ]:
# ========== PHASE X1.3: PREDICT IR ==========
app_train = pd.read_csv('data/raw/application_train.csv')
app_test = pd.read_csv('data/raw/application_test.csv')

app_train['CNT_PAYMENT_PROXY'] = app_train['AMT_CREDIT'] / (app_train['AMT_ANNUITY'] + 1)
app_test['CNT_PAYMENT_PROXY'] = app_test['AMT_CREDIT'] / (app_test['AMT_ANNUITY'] + 1)

def prepare_app_for_ir(df, cat_cols):
    result = pd.DataFrame()
    for f in prev_features:
        if f == 'CNT_PAYMENT':
            result[f] = df['CNT_PAYMENT_PROXY'].clip(0, 120).values
        elif f in df.columns:
            result[f] = df[f].values
        else:
            result[f] = np.nan

    for col in cat_cols:
        if col in result.columns:
            result[col] = result[col].fillna('MISSING').astype(str)
            codes, _ = pd.factorize(result[col])
            result[col] = codes.astype(np.int32)

    result = result.replace([np.inf, -np.inf], np.nan)
    return result

X_app_train = prepare_app_for_ir(app_train, cat_cols_ir)
X_app_test = prepare_app_for_ir(app_test, cat_cols_ir)

predicted_ir_train = np.zeros(len(app_train), dtype=np.float32)
predicted_ir_test = np.zeros(len(app_test), dtype=np.float32)

for model in ir_models:
    predicted_ir_train += model.predict(X_app_train, num_iteration=model.best_iteration) / len(ir_models)
    predicted_ir_test += model.predict(X_app_test, num_iteration=model.best_iteration) / len(ir_models)

print(f'Predicted IR train: mean={predicted_ir_train.mean():.4f}, std={predicted_ir_train.std():.4f}')

ir_train_map = pd.DataFrame({'SK_ID_CURR': app_train['SK_ID_CURR'].values, 'PREDICTED_IR': predicted_ir_train})
ir_test_map = pd.DataFrame({'SK_ID_CURR': app_test['SK_ID_CURR'].values, 'PREDICTED_IR': predicted_ir_test})

# Reload train V8
train = pd.read_parquet('train_merged_v8.parquet')
test = pd.read_parquet('test_merged_v8.parquet')

train = train.merge(ir_train_map, on='SK_ID_CURR', how='left')
test = test.merge(ir_test_map, on='SK_ID_CURR', how='left')

# Save standalone
ir_train_map.to_parquet(cfg.OUTPUT_DIR / 'feature_predicted_ir_train.parquet', index=False)
ir_test_map.to_parquet(cfg.OUTPUT_DIR / 'feature_predicted_ir_test.parquet', index=False)

# Save V9 parquets
train.to_parquet('train_merged_v9.parquet', index=False)
test.to_parquet('test_merged_v9.parquet', index=False)

from scipy.stats import spearmanr
y_arr = train[cfg.TARGET].astype(int).values
# Use the 'PREDICTED_IR' column from the merged 'train' DataFrame
corr_ir = spearmanr(train['PREDICTED_IR'].values, y_arr)[0]
print(f'Spearman correlation PREDICTED_IR with TARGET: {corr_ir:.4f}')

del app_train, app_test, prev_app, ir_train_data, X_app_train, X_app_test, X_ir
gc.collect()

print('V9 parquets saved.')

Predicted IR train: mean=0.1912, std=0.0348
Spearman correlation PREDICTED_IR with TARGET: 0.0171
V9 parquets saved.


### **Predicted EXT_SOURCE NULL fill**

In [ ]:
# ========== PHASE X1b.1: ANALYZE EXT_SOURCE NULLS ==========
import numpy as np
import pandas as pd
import gc

# Reload V9 (đã có PREDICTED_IR)
train = pd.read_parquet('train_merged_v9.parquet')
test = pd.read_parquet('test_merged_v9.parquet')

print(f'Train: {train.shape}, Test: {test.shape}')

# Check NULL counts cho EXT_SOURCE
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    null_train = train[col].isna().sum()
    null_test = test[col].isna().sum()
    print(f'{col}: train NULL = {null_train} ({null_train/len(train)*100:.1f}%), '
          f'test NULL = {null_test} ({null_test/len(test)*100:.1f}%)')

Train: (307507, 644), Test: (48744, 643)
EXT_SOURCE_1: train NULL = 173376 (56.4%), test NULL = 20532 (42.1%)
EXT_SOURCE_2: train NULL = 660 (0.2%), test NULL = 8 (0.0%)
EXT_SOURCE_3: train NULL = 60965 (19.8%), test NULL = 8668 (17.8%)


In [ ]:
# ========== PHASE X1b.2: PREDICT EXT_SOURCE_1 ==========
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
import time, gc

TARGET_EXT = 'EXT_SOURCE_1'  # Highest NULL rate

# Features dùng predict EXT_SOURCE_1 (không dùng các EXT_SOURCE khác)
EXT_PREDICTOR_FEATURES = [
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL',
    'CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE',
    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'OWN_CAR_AGE',
    'CREDIT_TERM', 'CREDIT_GOODS_RATIO', 'PAYMENT_RATE',
]
EXT_PREDICTOR_FEATURES = [f for f in EXT_PREDICTOR_FEATURES if f in train.columns]
print(f'Predictor features: {len(EXT_PREDICTOR_FEATURES)}')

# Build training data: rows with EXT_SOURCE_1 not null
combined = pd.concat([train, test], ignore_index=True)
combined['is_test'] = [0]*len(train) + [1]*len(test)

ext_train_mask = combined[TARGET_EXT].notna()
ext_train = combined[ext_train_mask].copy()
ext_test = combined[~ext_train_mask].copy()

print(f'EXT_SOURCE_1 training samples: {len(ext_train)}')
print(f'EXT_SOURCE_1 to predict:       {len(ext_test)}')

X_ext = ext_train[EXT_PREDICTOR_FEATURES].copy()
y_ext = ext_train[TARGET_EXT].values
X_ext_pred = ext_test[EXT_PREDICTOR_FEATURES].copy()

cat_cols_ext = []
for col in EXT_PREDICTOR_FEATURES:
    if X_ext[col].dtype == 'object':
        X_ext[col] = X_ext[col].fillna('MISSING').astype(str)
        X_ext_pred[col] = X_ext_pred[col].fillna('MISSING').astype(str)
        combined_vals = pd.concat([X_ext[col], X_ext_pred[col]])
        codes, _ = pd.factorize(combined_vals)
        X_ext[col] = codes[:len(X_ext)].astype(np.int32)
        X_ext_pred[col] = codes[len(X_ext):].astype(np.int32)
        cat_cols_ext.append(col)

X_ext = X_ext.replace([np.inf, -np.inf], np.nan)
X_ext_pred = X_ext_pred.replace([np.inf, -np.inf], np.nan)

LGB_EXT_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6,
    'min_child_samples': 100, 'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'colsample_bytree': 0.8, 'subsample': 0.8, 'subsample_freq': 1,
    'verbosity': -1, 'random_state': 42, 'n_jobs': -1,
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ext_models = []
rmse_scores = []

print(f'Training EXT_SOURCE_1 predictor...')
print('=' * 70)
start = time.time()

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_ext)):
    X_tr, X_val = X_ext.iloc[tr_idx], X_ext.iloc[val_idx]
    y_tr, y_val = y_ext[tr_idx], y_ext[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols_ext)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols_ext)

    model = lgb.train(
        LGB_EXT_PARAMS, dtrain, num_boost_round=2000,
        valid_sets=[dval], valid_names=['valid'],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    r2 = r2_score(y_val, val_preds)

    rmse_scores.append(rmse)
    ext_models.append(model)
    print(f'Fold {fold_idx+1}/5 | RMSE: {rmse:.4f} | R²: {r2:.4f}')
    del dtrain, dval
    gc.collect()

print(f'\nMean RMSE: {np.mean(rmse_scores):.4f}')
print(f'Time: {time.time()-start:.0f}s')

# Predict missing values
predicted_ext1 = np.zeros(len(X_ext_pred), dtype=np.float32)
for model in ext_models:
    predicted_ext1 += model.predict(X_ext_pred, num_iteration=model.best_iteration) / len(ext_models)

# Map back
ext_test['PREDICTED_EXT_SOURCE_1'] = predicted_ext1
ext_train['PREDICTED_EXT_SOURCE_1'] = ext_train[TARGET_EXT].values  # Original values for non-null

combined['PREDICTED_EXT_SOURCE_1'] = np.nan
combined.loc[ext_train_mask, 'PREDICTED_EXT_SOURCE_1'] = ext_train['PREDICTED_EXT_SOURCE_1'].values
combined.loc[~ext_train_mask, 'PREDICTED_EXT_SOURCE_1'] = ext_test['PREDICTED_EXT_SOURCE_1'].values

# Split back train/test
train['PREDICTED_EXT_SOURCE_1'] = combined.iloc[:len(train)]['PREDICTED_EXT_SOURCE_1'].values.astype(np.float32)
test['PREDICTED_EXT_SOURCE_1'] = combined.iloc[len(train):]['PREDICTED_EXT_SOURCE_1'].values.astype(np.float32)

print(f'\nPREDICTED_EXT_SOURCE_1 stats:')
print(f'  Train: mean={train["PREDICTED_EXT_SOURCE_1"].mean():.4f}, NaN={train["PREDICTED_EXT_SOURCE_1"].isna().sum()}')
print(f'  Test:  mean={test["PREDICTED_EXT_SOURCE_1"].mean():.4f}, NaN={test["PREDICTED_EXT_SOURCE_1"].isna().sum()}')

del combined, ext_train, ext_test, ext_models, X_ext, X_ext_pred
gc.collect()

Predictor features: 25
EXT_SOURCE_1 training samples: 162343
EXT_SOURCE_1 to predict:       193908
Training EXT_SOURCE_1 predictor...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[427]	valid's rmse: 0.153478
Fold 1/5 | RMSE: 0.1535 | R²: 0.4689
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[529]	valid's rmse: 0.152568
Fold 2/5 | RMSE: 0.1526 | R²: 0.4779
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[443]	valid's rmse: 0.153029
Fold 3/5 | RMSE: 0.1530 | R²: 0.4653
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[476]	valid's rmse: 0.152023
Fold 4/5 | RMSE: 0.1520 | R²: 0.4744
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[508]	valid's rmse: 0.152575
Fold 5/5 | RMSE: 0.1526 | R²: 0.4696

Mean RMSE: 0.1527
Time: 29s

PREDICTED_EXT_SOURCE_

20

In [ ]:
# ========== PHASE X1b.3: PREDICT EXT_SOURCE_3 ==========
TARGET_EXT = 'EXT_SOURCE_3'  # Second highest NULL

combined = pd.concat([train, test], ignore_index=True)
ext_train_mask = combined[TARGET_EXT].notna()
ext_train = combined[ext_train_mask].copy()
ext_test = combined[~ext_train_mask].copy()

print(f'EXT_SOURCE_3 training samples: {len(ext_train)}')
print(f'EXT_SOURCE_3 to predict:       {len(ext_test)}')

X_ext = ext_train[EXT_PREDICTOR_FEATURES].copy()
y_ext = ext_train[TARGET_EXT].values
X_ext_pred = ext_test[EXT_PREDICTOR_FEATURES].copy()

cat_cols_ext = []
for col in EXT_PREDICTOR_FEATURES:
    if X_ext[col].dtype == 'object':
        X_ext[col] = X_ext[col].fillna('MISSING').astype(str)
        X_ext_pred[col] = X_ext_pred[col].fillna('MISSING').astype(str)
        combined_vals = pd.concat([X_ext[col], X_ext_pred[col]])
        codes, _ = pd.factorize(combined_vals)
        X_ext[col] = codes[:len(X_ext)].astype(np.int32)
        X_ext_pred[col] = codes[len(X_ext):].astype(np.int32)
        cat_cols_ext.append(col)

X_ext = X_ext.replace([np.inf, -np.inf], np.nan)
X_ext_pred = X_ext_pred.replace([np.inf, -np.inf], np.nan)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ext_models = []
rmse_scores = []

print(f'Training EXT_SOURCE_3 predictor...')
print('=' * 70)
start = time.time()

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_ext)):
    X_tr, X_val = X_ext.iloc[tr_idx], X_ext.iloc[val_idx]
    y_tr, y_val = y_ext[tr_idx], y_ext[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols_ext)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols_ext)

    model = lgb.train(
        LGB_EXT_PARAMS, dtrain, num_boost_round=2000,
        valid_sets=[dval], valid_names=['valid'],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    rmse_scores.append(rmse)
    ext_models.append(model)
    print(f'Fold {fold_idx+1}/5 | RMSE: {rmse:.4f}')
    del dtrain, dval
    gc.collect()

print(f'Mean RMSE: {np.mean(rmse_scores):.4f}, Time: {time.time()-start:.0f}s')

predicted_ext3 = np.zeros(len(X_ext_pred), dtype=np.float32)
for model in ext_models:
    predicted_ext3 += model.predict(X_ext_pred, num_iteration=model.best_iteration) / len(ext_models)

ext_test['PREDICTED_EXT_SOURCE_3'] = predicted_ext3
ext_train['PREDICTED_EXT_SOURCE_3'] = ext_train[TARGET_EXT].values

combined['PREDICTED_EXT_SOURCE_3'] = np.nan
combined.loc[ext_train_mask, 'PREDICTED_EXT_SOURCE_3'] = ext_train['PREDICTED_EXT_SOURCE_3'].values
combined.loc[~ext_train_mask, 'PREDICTED_EXT_SOURCE_3'] = ext_test['PREDICTED_EXT_SOURCE_3'].values

train['PREDICTED_EXT_SOURCE_3'] = combined.iloc[:len(train)]['PREDICTED_EXT_SOURCE_3'].values.astype(np.float32)
test['PREDICTED_EXT_SOURCE_3'] = combined.iloc[len(train):]['PREDICTED_EXT_SOURCE_3'].values.astype(np.float32)

# Save V10
train.to_parquet('train_merged_v10.parquet', index=False)
test.to_parquet('test_merged_v10.parquet', index=False)

del combined, ext_train, ext_test, ext_models
gc.collect()

print('V10 parquets saved (with PREDICTED_EXT_SOURCE_1 + PREDICTED_EXT_SOURCE_3)')

EXT_SOURCE_3 training samples: 286618
EXT_SOURCE_3 to predict:       69633
Training EXT_SOURCE_3 predictor...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[761]	valid's rmse: 0.18182
Fold 1/5 | RMSE: 0.1818
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[766]	valid's rmse: 0.181283
Fold 2/5 | RMSE: 0.1813
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[838]	valid's rmse: 0.181774
Fold 3/5 | RMSE: 0.1818
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[766]	valid's rmse: 0.182085
Fold 4/5 | RMSE: 0.1821
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[760]	valid's rmse: 0.181873
Fold 5/5 | RMSE: 0.1819
Mean RMSE: 0.1818, Time: 64s
V10 parquets saved (with PREDICTED_EXT_SOURCE_1 + PREDICTED_EXT_SOURCE_3)


### Bureau / previous cross-interactions

Cross-interaction features combine current application attributes with external credit history, such as bureau debt relative to current credit amount.

Heavy cell: do not run during demo unless local artifacts and sufficient runtime are available.


### **BUREAU/PREV CROSS-INTERACTIONS**

In [ ]:
# ========== PHASE 1: BUREAU/PREV CROSS-INTERACTIONS ==========
import numpy as np
import pandas as pd
import gc

# Reload V10 (đã có TARGET_NN500, PREDICTED_IR, PREDICTED_EXT_SOURCE_1/3)
train = pd.read_parquet('train_merged_v10.parquet')
test = pd.read_parquet('test_merged_v10.parquet')

print(f'V10: train={train.shape}, test={test.shape}')

def add_cross_features(df):
    """Add cross-table interactions."""
    eps = 1e-5

    # === Bureau × Application interactions ===
    if 'bur_AMT_CREDIT_SUM_DEBT_mean' in df.columns and 'AMT_CREDIT' in df.columns:
        df['BUR_DEBT_TO_CURRENT_CREDIT'] = df['bur_AMT_CREDIT_SUM_DEBT_mean'] / (df['AMT_CREDIT'] + eps)

    if 'bur_AMT_CREDIT_SUM_mean' in df.columns and 'AMT_CREDIT' in df.columns:
        df['BUR_CREDIT_TO_CURRENT_CREDIT'] = df['bur_AMT_CREDIT_SUM_mean'] / (df['AMT_CREDIT'] + eps)

    if 'bur_AMT_ANNUITY_mean' in df.columns and 'AMT_ANNUITY' in df.columns:
        df['BUR_ANNUITY_TO_CURRENT_ANNUITY'] = df['bur_AMT_ANNUITY_mean'] / (df['AMT_ANNUITY'] + eps)

    if 'bur_DAYS_CREDIT_mean' in df.columns and 'DAYS_BIRTH' in df.columns:
        df['BUR_DAYS_CREDIT_TO_AGE'] = df['bur_DAYS_CREDIT_mean'] / (df['DAYS_BIRTH'] - eps)

    if 'bur_count' in df.columns and 'DAYS_BIRTH' in df.columns:
        df['BUR_COUNT_PER_YEAR'] = df['bur_count'] / ((-df['DAYS_BIRTH'] / 365.25) + eps)

    # === Previous Application × Application interactions ===
    if 'pa_AMT_CREDIT_mean' in df.columns and 'AMT_CREDIT' in df.columns:
        df['PA_CREDIT_TO_CURRENT_CREDIT'] = df['pa_AMT_CREDIT_mean'] / (df['AMT_CREDIT'] + eps)

    if 'pa_AMT_APPLICATION_mean' in df.columns and 'AMT_CREDIT' in df.columns:
        df['PA_APPLICATION_TO_CURRENT_CREDIT'] = df['pa_AMT_APPLICATION_mean'] / (df['AMT_CREDIT'] + eps)

    if 'pa_AMT_ANNUITY_mean' in df.columns and 'AMT_ANNUITY' in df.columns:
        df['PA_ANNUITY_TO_CURRENT_ANNUITY'] = df['pa_AMT_ANNUITY_mean'] / (df['AMT_ANNUITY'] + eps)

    if 'pa_count' in df.columns and 'bur_count' in df.columns:
        df['PA_BUR_COUNT_RATIO'] = df['pa_count'] / (df['bur_count'] + eps)
        df['PA_BUR_COUNT_DIFF'] = df['pa_count'] - df['bur_count']

    # === Installment × Current interactions ===
    if 'ip_total_actual_payment' in df.columns and 'AMT_CREDIT' in df.columns:
        df['INST_PAYMENT_TO_CURRENT_CREDIT'] = df['ip_total_actual_payment'] / (df['AMT_CREDIT'] + eps)

    if 'ip_total_late_days_1y' in df.columns and 'DAYS_BIRTH' in df.columns:
        df['INST_LATE_DAYS_PER_AGE'] = df['ip_total_late_days_1y'] / ((-df['DAYS_BIRTH'] / 365.25) + eps)

    # === Credit Card × Current ===
    if 'cc_AMT_BALANCE_mean' in df.columns and 'AMT_CREDIT' in df.columns:
        df['CC_BALANCE_TO_CURRENT_CREDIT'] = df['cc_AMT_BALANCE_mean'] / (df['AMT_CREDIT'] + eps)

    # === Cross-source: total debt picture ===
    debt_components = []
    if 'bur_AMT_CREDIT_SUM_DEBT_mean' in df.columns:
        debt_components.append(df['bur_AMT_CREDIT_SUM_DEBT_mean'].fillna(0))
    if 'cc_AMT_BALANCE_mean' in df.columns:
        debt_components.append(df['cc_AMT_BALANCE_mean'].fillna(0))

    if len(debt_components) >= 2:
        df['TOTAL_EXTERNAL_DEBT'] = sum(debt_components)
        df['TOTAL_EXTERNAL_DEBT_TO_INCOME'] = df['TOTAL_EXTERNAL_DEBT'] / (df['AMT_INCOME_TOTAL'] + eps)

    # === Behavioral patterns ===
    if 'PREDICTED_EXT_SOURCE_1' in df.columns and 'EXT_SOURCE_2' in df.columns:
        df['EXT1_PREDICTED_x_EXT2'] = df['PREDICTED_EXT_SOURCE_1'] * df['EXT_SOURCE_2']

    if 'PREDICTED_EXT_SOURCE_3' in df.columns and 'EXT_SOURCE_2' in df.columns:
        df['EXT3_PREDICTED_x_EXT2'] = df['PREDICTED_EXT_SOURCE_3'] * df['EXT_SOURCE_2']

    # === TARGET_NN500 interactions ===
    if 'TARGET_NN500_MEAN' in df.columns and 'EXT_MEAN' in df.columns:
        df['NN500_x_EXT_MEAN'] = df['TARGET_NN500_MEAN'] * df['EXT_MEAN']

    if 'TARGET_NN500_MEAN' in df.columns and 'CREDIT_TERM' in df.columns:
        df['NN500_x_CREDIT_TERM'] = df['TARGET_NN500_MEAN'] * df['CREDIT_TERM']

    return df

print('Adding cross-interaction features...')
train = add_cross_features(train)
test = add_cross_features(test)

print(f'V11: train={train.shape}, test={test.shape}')
new_features = [c for c in train.columns if c not in pd.read_parquet('train_merged_v10.parquet').columns]
print(f'New features added: {len(new_features)}')
for f in new_features:
    print(f'  {f}')

train.to_parquet('train_merged_v11.parquet', index=False)
test.to_parquet('test_merged_v11.parquet', index=False)
print('\nV11 parquets saved.')

del new_features
gc.collect()

V10: train=(307507, 646), test=(48744, 645)
Adding cross-interaction features...
V11: train=(307507, 652), test=(48744, 651)
New features added: 6
  INST_PAYMENT_TO_CURRENT_CREDIT
  INST_LATE_DAYS_PER_AGE
  EXT1_PREDICTED_x_EXT2
  EXT3_PREDICTED_x_EXT2
  NN500_x_EXT_MEAN
  NN500_x_CREDIT_TERM

V11 parquets saved.


0

### Time-window and loan-type aggregations

Time-window features capture recency effects, while loan-type features separate behavior across credit cards, consumer credit, mortgages, microloans, and other loan types.

Heavy cells: do not run during demo unless local artifacts and sufficient runtime are available.


### **more features**

In [ ]:
# ========== PHASE K1: TIME-WINDOW AGGREGATIONS ==========
import pandas as pd
import numpy as np
import gc
import time

start_time = time.time()

# Load raw CSV để aggregate fresh
print('Loading raw tables...')
bureau = pd.read_csv('data/raw/bureau.csv')
prev_app = pd.read_csv('data/raw/previous_application.csv')
installments = pd.read_csv('data/raw/installments_payments.csv')
pos_cash = pd.read_csv('data/raw/POS_CASH_balance.csv')
credit_card = pd.read_csv('data/raw/credit_card_balance.csv')

# Load V11 dataset
train_v11 = pd.read_parquet('train_merged_v11.parquet')
test_v11 = pd.read_parquet('test_merged_v11.parquet')

print(f'V11 base: train={train_v11.shape}, test={test_v11.shape}')

# ===== 1. BUREAU TIME-WINDOW =====
print('\n[1/5] Bureau time-window aggregations...')

# DAYS_CREDIT là negative (more negative = older)
bureau_recent_3m = bureau[bureau['DAYS_CREDIT'] >= -90].copy()
bureau_recent_6m = bureau[bureau['DAYS_CREDIT'] >= -180].copy()
bureau_recent_12m = bureau[bureau['DAYS_CREDIT'] >= -365].copy()
bureau_recent_24m = bureau[bureau['DAYS_CREDIT'] >= -730].copy()
bureau_old_24m = bureau[bureau['DAYS_CREDIT'] < -730].copy()

bureau_agg_cols = ['AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_MAX_OVERDUE',
                   'CNT_CREDIT_PROLONG', 'DAYS_CREDIT_UPDATE']

def aggregate_time_window(df, suffix, cols=bureau_agg_cols):
    agg = df.groupby('SK_ID_CURR').agg({
        col: ['mean', 'max', 'sum'] for col in cols if col in df.columns
    })
    agg.columns = [f'BUR_{c[0]}_{c[1]}_{suffix}' for c in agg.columns]
    agg[f'BUR_count_{suffix}'] = df.groupby('SK_ID_CURR').size()
    return agg.reset_index()

bur_3m = aggregate_time_window(bureau_recent_3m, '3m')
bur_6m = aggregate_time_window(bureau_recent_6m, '6m')
bur_12m = aggregate_time_window(bureau_recent_12m, '12m')
bur_24m = aggregate_time_window(bureau_recent_24m, '24m')
bur_old = aggregate_time_window(bureau_old_24m, 'old')

print(f'  Bureau 3m features: {bur_3m.shape[1] - 1}')
print(f'  Bureau 6m features: {bur_6m.shape[1] - 1}')
print(f'  Bureau 12m features: {bur_12m.shape[1] - 1}')

# Merge into train/test
for agg_df in [bur_3m, bur_6m, bur_12m, bur_24m, bur_old]:
    train_v11 = train_v11.merge(agg_df, on='SK_ID_CURR', how='left')
    test_v11 = test_v11.merge(agg_df, on='SK_ID_CURR', how='left')

del bureau_recent_3m, bureau_recent_6m, bureau_recent_12m, bureau_recent_24m, bureau_old_24m
del bur_3m, bur_6m, bur_12m, bur_24m, bur_old, bureau
gc.collect()

print(f'  After bureau time-window: train={train_v11.shape}, test={test_v11.shape}')

# ===== 2. PREVIOUS APPLICATION TIME-WINDOW =====
print('\n[2/5] Previous Application time-window aggregations...')

prev_recent_3m = prev_app[prev_app['DAYS_DECISION'] >= -90].copy()
prev_recent_6m = prev_app[prev_app['DAYS_DECISION'] >= -180].copy()
prev_recent_12m = prev_app[prev_app['DAYS_DECISION'] >= -365].copy()
prev_recent_24m = prev_app[prev_app['DAYS_DECISION'] >= -730].copy()

prev_agg_cols = ['AMT_APPLICATION', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_DOWN_PAYMENT',
                 'CNT_PAYMENT', 'RATE_DOWN_PAYMENT', 'DAYS_DECISION']

def aggregate_prev_time_window(df, suffix):
    agg = df.groupby('SK_ID_CURR').agg({
        col: ['mean', 'max', 'sum'] for col in prev_agg_cols if col in df.columns
    })
    agg.columns = [f'PA_{c[0]}_{c[1]}_{suffix}' for c in agg.columns]
    agg[f'PA_count_{suffix}'] = df.groupby('SK_ID_CURR').size()
    return agg.reset_index()

prev_3m = aggregate_prev_time_window(prev_recent_3m, '3m')
prev_6m = aggregate_prev_time_window(prev_recent_6m, '6m')
prev_12m = aggregate_prev_time_window(prev_recent_12m, '12m')
prev_24m = aggregate_prev_time_window(prev_recent_24m, '24m')

for agg_df in [prev_3m, prev_6m, prev_12m, prev_24m]:
    train_v11 = train_v11.merge(agg_df, on='SK_ID_CURR', how='left')
    test_v11 = test_v11.merge(agg_df, on='SK_ID_CURR', how='left')

del prev_recent_3m, prev_recent_6m, prev_recent_12m, prev_recent_24m
del prev_3m, prev_6m, prev_12m, prev_24m
gc.collect()

print(f'  After prev_app time-window: train={train_v11.shape}, test={test_v11.shape}')

# ===== 3. INSTALLMENTS TIME-WINDOW =====
print('\n[3/5] Installments time-window aggregations...')

# DAYS_INSTALMENT là negative (more negative = older)
inst_3m = installments[installments['DAYS_INSTALMENT'] >= -90].copy()
inst_6m = installments[installments['DAYS_INSTALMENT'] >= -180].copy()
inst_12m = installments[installments['DAYS_INSTALMENT'] >= -365].copy()

# DPD calculation
for inst_df in [inst_3m, inst_6m, inst_12m]:
    inst_df['DPD'] = inst_df['DAYS_ENTRY_PAYMENT'] - inst_df['DAYS_INSTALMENT']
    inst_df['DPD'] = inst_df['DPD'].apply(lambda x: x if x > 0 else 0)
    inst_df['PAYMENT_RATIO'] = inst_df['AMT_PAYMENT'] / inst_df['AMT_INSTALMENT']
    inst_df['PAYMENT_DIFF'] = inst_df['AMT_INSTALMENT'] - inst_df['AMT_PAYMENT']

def aggregate_inst_time_window(df, suffix):
    agg = df.groupby('SK_ID_CURR').agg({
        'DPD': ['mean', 'max', 'sum'],
        'PAYMENT_RATIO': ['mean', 'min'],
        'PAYMENT_DIFF': ['mean', 'max', 'sum'],
        'AMT_PAYMENT': ['mean', 'sum'],
    })
    agg.columns = [f'INST_{c[0]}_{c[1]}_{suffix}' for c in agg.columns]
    agg[f'INST_count_{suffix}'] = df.groupby('SK_ID_CURR').size()
    return agg.reset_index()

inst_agg_3m = aggregate_inst_time_window(inst_3m, '3m')
inst_agg_6m = aggregate_inst_time_window(inst_6m, '6m')
inst_agg_12m = aggregate_inst_time_window(inst_12m, '12m')

for agg_df in [inst_agg_3m, inst_agg_6m, inst_agg_12m]:
    train_v11 = train_v11.merge(agg_df, on='SK_ID_CURR', how='left')
    test_v11 = test_v11.merge(agg_df, on='SK_ID_CURR', how='left')

del inst_3m, inst_6m, inst_12m, inst_agg_3m, inst_agg_6m, inst_agg_12m, installments
gc.collect()

print(f'  After installments time-window: train={train_v11.shape}, test={test_v11.shape}')

# ===== 4. POS_CASH TIME-WINDOW =====
print('\n[4/5] POS_CASH time-window aggregations...')

# MONTHS_BALANCE là negative
pos_3m = pos_cash[pos_cash['MONTHS_BALANCE'] >= -3].copy()
pos_6m = pos_cash[pos_cash['MONTHS_BALANCE'] >= -6].copy()
pos_12m = pos_cash[pos_cash['MONTHS_BALANCE'] >= -12].copy()

def aggregate_pos_time_window(df, suffix):
    agg = df.groupby('SK_ID_CURR').agg({
        'CNT_INSTALMENT': ['mean', 'max'],
        'CNT_INSTALMENT_FUTURE': ['mean', 'max'],
        'SK_DPD': ['mean', 'max', 'sum'],
        'SK_DPD_DEF': ['mean', 'max'],
    })
    agg.columns = [f'POS_{c[0]}_{c[1]}_{suffix}' for c in agg.columns]
    return agg.reset_index()

pos_agg_3m = aggregate_pos_time_window(pos_3m, '3m')
pos_agg_6m = aggregate_pos_time_window(pos_6m, '6m')
pos_agg_12m = aggregate_pos_time_window(pos_12m, '12m')

for agg_df in [pos_agg_3m, pos_agg_6m, pos_agg_12m]:
    train_v11 = train_v11.merge(agg_df, on='SK_ID_CURR', how='left')
    test_v11 = test_v11.merge(agg_df, on='SK_ID_CURR', how='left')

del pos_3m, pos_6m, pos_12m, pos_agg_3m, pos_agg_6m, pos_agg_12m, pos_cash
gc.collect()

# ===== 5. CREDIT CARD TIME-WINDOW =====
print('\n[5/5] Credit Card time-window aggregations...')

cc_3m = credit_card[credit_card['MONTHS_BALANCE'] >= -3].copy()
cc_6m = credit_card[credit_card['MONTHS_BALANCE'] >= -6].copy()
cc_12m = credit_card[credit_card['MONTHS_BALANCE'] >= -12].copy()

def aggregate_cc_time_window(df, suffix):
    agg = df.groupby('SK_ID_CURR').agg({
        'AMT_BALANCE': ['mean', 'max'],
        'AMT_CREDIT_LIMIT_ACTUAL': ['mean', 'max'],
        'AMT_DRAWINGS_CURRENT': ['mean', 'sum'],
        'AMT_PAYMENT_CURRENT': ['mean', 'sum'],
        'SK_DPD': ['mean', 'max'],
    })
    agg.columns = [f'CC_{c[0]}_{c[1]}_{suffix}' for c in agg.columns]
    return agg.reset_index()

cc_agg_3m = aggregate_cc_time_window(cc_3m, '3m')
cc_agg_6m = aggregate_cc_time_window(cc_6m, '6m')
cc_agg_12m = aggregate_cc_time_window(cc_12m, '12m')

for agg_df in [cc_agg_3m, cc_agg_6m, cc_agg_12m]:
    train_v11 = train_v11.merge(agg_df, on='SK_ID_CURR', how='left')
    test_v11 = test_v11.merge(agg_df, on='SK_ID_CURR', how='left')

del cc_3m, cc_6m, cc_12m, cc_agg_3m, cc_agg_6m, cc_agg_12m, credit_card
gc.collect()

print(f'\nAfter K1: train={train_v11.shape}, test={test_v11.shape}')
print(f'K1 elapsed: {(time.time() - start_time)/60:.1f} min')

# Save intermediate
train_v11.to_parquet('train_merged_v12_intermediate.parquet', index=False)
test_v11.to_parquet('test_merged_v12_intermediate.parquet', index=False)
print('Saved intermediate.')

Loading raw tables...
V11 base: train=(307507, 652), test=(48744, 651)

[1/5] Bureau time-window aggregations...
  Bureau 3m features: 16
  Bureau 6m features: 16
  Bureau 12m features: 16
  After bureau time-window: train=(307507, 732), test=(48744, 731)

[2/5] Previous Application time-window aggregations...
  After prev_app time-window: train=(307507, 820), test=(48744, 819)

[3/5] Installments time-window aggregations...
  After installments time-window: train=(307507, 853), test=(48744, 852)

[4/5] POS_CASH time-window aggregations...

[5/5] Credit Card time-window aggregations...

After K1: train=(307507, 910), test=(48744, 909)
K1 elapsed: 1.4 min
Saved intermediate.


In [ ]:
# ========== PHASE K2: LOAN-TYPE SPECIFIC AGGREGATIONS ==========
import pandas as pd
import numpy as np
import gc
import time

start_time = time.time()

# Reload tables
bureau = pd.read_csv('data/raw/bureau.csv')
prev_app = pd.read_csv('data/raw/previous_application.csv')

# Load intermediate
train_v12 = pd.read_parquet('train_merged_v12_intermediate.parquet')
test_v12 = pd.read_parquet('test_merged_v12_intermediate.parquet')

print(f'V12 intermediate: train={train_v12.shape}, test={test_v12.shape}')

# ===== 1. BUREAU BY CREDIT_TYPE =====
print('\n[1/3] Bureau by CREDIT_TYPE...')

# Top 4 credit types (Consumer credit, Credit card, Car loan, Mortgage)
top_credit_types = bureau['CREDIT_TYPE'].value_counts().head(4).index.tolist()
print(f'  Top credit types: {top_credit_types}')

bureau_cols = ['AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_MAX_OVERDUE',
               'DAYS_CREDIT', 'CNT_CREDIT_PROLONG']

for credit_type in top_credit_types:
    safe_name = credit_type.replace(' ', '_').replace('-', '_')
    bur_type = bureau[bureau['CREDIT_TYPE'] == credit_type]

    agg = bur_type.groupby('SK_ID_CURR').agg({
        col: ['mean', 'max', 'sum'] for col in bureau_cols if col in bur_type.columns
    })
    agg.columns = [f'BUR_{safe_name}_{c[0]}_{c[1]}' for c in agg.columns]
    agg[f'BUR_{safe_name}_count'] = bur_type.groupby('SK_ID_CURR').size()
    agg = agg.reset_index()

    train_v12 = train_v12.merge(agg, on='SK_ID_CURR', how='left')
    test_v12 = test_v12.merge(agg, on='SK_ID_CURR', how='left')
    print(f'  {credit_type}: {agg.shape[1] - 1} features')
    del bur_type, agg
    gc.collect()

# ===== 2. BUREAU BY CREDIT_ACTIVE =====
print('\n[2/3] Bureau by CREDIT_ACTIVE...')

for status in ['Active', 'Closed']:
    bur_status = bureau[bureau['CREDIT_ACTIVE'] == status]

    agg = bur_status.groupby('SK_ID_CURR').agg({
        col: ['mean', 'max', 'sum'] for col in bureau_cols if col in bur_status.columns
    })
    agg.columns = [f'BUR_{status}_{c[0]}_{c[1]}' for c in agg.columns]
    agg[f'BUR_{status}_count'] = bur_status.groupby('SK_ID_CURR').size()
    agg = agg.reset_index()

    train_v12 = train_v12.merge(agg, on='SK_ID_CURR', how='left')
    test_v12 = test_v12.merge(agg, on='SK_ID_CURR', how='left')
    print(f'  {status}: {agg.shape[1] - 1} features')
    del bur_status, agg
    gc.collect()

# ===== 3. PREVIOUS APP BY STATUS =====
print('\n[3/3] Previous App by NAME_CONTRACT_STATUS...')

prev_cols = ['AMT_APPLICATION', 'AMT_CREDIT', 'AMT_ANNUITY', 'CNT_PAYMENT', 'DAYS_DECISION']

for status in ['Approved', 'Refused', 'Canceled']:
    prev_status = prev_app[prev_app['NAME_CONTRACT_STATUS'] == status]

    agg = prev_status.groupby('SK_ID_CURR').agg({
        col: ['mean', 'max', 'sum'] for col in prev_cols if col in prev_status.columns
    })
    agg.columns = [f'PA_{status}_{c[0]}_{c[1]}' for c in agg.columns]
    agg[f'PA_{status}_count'] = prev_status.groupby('SK_ID_CURR').size()
    agg = agg.reset_index()

    train_v12 = train_v12.merge(agg, on='SK_ID_CURR', how='left')
    test_v12 = test_v12.merge(agg, on='SK_ID_CURR', how='left')
    print(f'  {status}: {agg.shape[1] - 1} features')
    del prev_status, agg
    gc.collect()

del bureau, prev_app
gc.collect()

# Compute important RATIOS between loan-type groups
print('\n[BONUS] Computing loan-type ratios...')

eps = 1e-5

if 'PA_Approved_count' in train_v12.columns and 'PA_Refused_count' in train_v12.columns:
    train_v12['PA_REFUSED_RATIO'] = train_v12['PA_Refused_count'] / (train_v12['PA_Approved_count'] + train_v12['PA_Refused_count'] + eps)
    test_v12['PA_REFUSED_RATIO'] = test_v12['PA_Refused_count'] / (test_v12['PA_Approved_count'] + test_v12['PA_Refused_count'] + eps)

if 'BUR_Active_count' in train_v12.columns and 'BUR_Closed_count' in train_v12.columns:
    train_v12['BUR_ACTIVE_RATIO'] = train_v12['BUR_Active_count'] / (train_v12['BUR_Active_count'] + train_v12['BUR_Closed_count'] + eps)
    test_v12['BUR_ACTIVE_RATIO'] = test_v12['BUR_Active_count'] / (test_v12['BUR_Active_count'] + test_v12['BUR_Closed_count'] + eps)

print(f'\nAfter K2: train={train_v12.shape}, test={test_v12.shape}')
print(f'K2 elapsed: {(time.time() - start_time)/60:.1f} min')

# Save final V12
train_v12.to_parquet('train_merged_v12.parquet', index=False)
test_v12.to_parquet('test_merged_v12.parquet', index=False)

new_features = [c for c in train_v12.columns if c not in pd.read_parquet('train_merged_v11.parquet').columns]
print(f'\nTotal new features (V12 vs V11): {len(new_features)}')
print('V12 saved.')

del train_v12, test_v12
gc.collect()

V12 intermediate: train=(307507, 910), test=(48744, 909)

[1/3] Bureau by CREDIT_TYPE...
  Top credit types: ['Consumer credit', 'Credit card', 'Car loan', 'Mortgage']
  Consumer credit: 16 features
  Credit card: 16 features
  Car loan: 16 features
  Mortgage: 16 features

[2/3] Bureau by CREDIT_ACTIVE...
  Active: 16 features
  Closed: 16 features

[3/3] Previous App by NAME_CONTRACT_STATUS...
  Approved: 16 features
  Refused: 16 features
  Canceled: 16 features

[BONUS] Computing loan-type ratios...

After K2: train=(307507, 1055), test=(48744, 1054)
K2 elapsed: 0.6 min

Total new features (V12 vs V11): 403
V12 saved.


0

### Stability filtering / V14 cleanup

This stage checks train-test distribution shift and drops unstable predictors while protecting key high-signal features.

Heavy cells: do not run during demo unless local artifacts and sufficient runtime are available.


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import gc, time
import warnings
warnings.filterwarnings('ignore')

start_time = time.time()

# Load V12 (1053 features - best feature set)
print('Loading V12 dataset...')
train = pd.read_parquet('train_merged_v12.parquet')
test = pd.read_parquet('test_merged_v12.parquet')

print(f'Train: {train.shape}, Test: {test.shape}')

# Prepare adversarial dataset
EXCLUDE_COLS = ['SK_ID_CURR', cfg.TARGET, 'index']
feature_cols = [c for c in train.columns if c not in EXCLUDE_COLS]

# Add is_test label
train_adv = train[feature_cols].copy()
train_adv['is_test'] = 0
test_adv = test[feature_cols].copy()
test_adv['is_test'] = 1

# Encode categoricals
for col in feature_cols:
    if train_adv[col].dtype == 'object':
        combined_col = pd.concat([train_adv[col].astype(str), test_adv[col].astype(str)])
        codes, _ = pd.factorize(combined_col)
        train_adv[col] = codes[:len(train_adv)].astype(np.int32)
        test_adv[col] = codes[len(train_adv):].astype(np.int32)

# Combine
adv_data = pd.concat([train_adv, test_adv], axis=0).reset_index(drop=True)
y_adv = adv_data['is_test'].values
X_adv = adv_data[feature_cols]

print(f'Adversarial dataset: {X_adv.shape}, positive rate: {y_adv.mean():.3f}')

FORCE_CATEGORICAL = [
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'WEEKDAY_APPR_PROCESS_START',
    'HOUR_APPR_PROCESS_START', 'BURO_CLU_LABEL', 'PREV_CLU_LABEL',
]
categorical_features = [c for c in feature_cols if c in FORCE_CATEGORICAL]

ADV_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 32, 'max_depth': 8,
    'min_child_samples': 100, 'reg_alpha': 0.5, 'reg_lambda': 0.5,
    'colsample_bytree': 0.7, 'subsample': 0.8, 'subsample_freq': 1,
    'verbosity': -1, 'random_state': 42, 'n_jobs': -1,
}

N_FOLDS = 5
oof_adv = np.zeros(len(X_adv))
fi_list = []
fold_aucs = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
print('\nTraining adversarial classifier...')
print('=' * 70)

for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_adv, y_adv)):
    fold_start = time.time()
    X_tr, X_val = X_adv.iloc[tr_idx], X_adv.iloc[val_idx]
    y_tr, y_val = y_adv[tr_idx], y_adv[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_features)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_features)

    model = lgb.train(
        ADV_PARAMS, dtrain, num_boost_round=2000,
        valid_sets=[dval], valid_names=['valid'],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    oof_adv[val_idx] = val_preds

    val_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(val_auc)

    fi_list.append(pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importance(importance_type='gain'),
    }))

    print(f'Fold {fold_idx+1}/5 | adv AUC: {val_auc:.5f} | best_iter: {model.best_iteration} | time: {time.time()-fold_start:.1f}s')
    del model, dtrain, dval
    gc.collect()

adv_auc = roc_auc_score(y_adv, oof_adv)
print('=' * 70)
print(f'\n ADVERSARIAL VALIDATION RESULTS:')
print(f'  Mean fold AUC:  {np.mean(fold_aucs):.5f}')
print(f'  Std fold AUC:   {np.std(fold_aucs):.5f}')
print(f'  Overall AUC:    {adv_auc:.5f}')
print(f'')

# === Determine severity based on adv_auc ===
if adv_auc > 0.95:
    severity = 'SEVERE'
elif adv_auc > 0.85:
    severity = 'SIGNIFICANT'
elif adv_auc > 0.70:
    severity = 'MODERATE'
else:
    severity = 'LOW'

# === Feature Stability Analysis ===
fi_avg = pd.concat(fi_list).groupby('feature')['importance'].agg(['mean', 'std']).reset_index()
fi_avg['cv'] = fi_avg['std'] / (fi_avg['mean'] + 1e-10)
fi_avg = fi_avg.sort_values('mean', ascending=False).reset_index(drop=True)

print(f'\n TOP 30 UNSTABLE FEATURES (high adversarial importance):')
print('(These features differentiate train vs test = unstable)')
print('-' * 70)
print(fi_avg.head(30)[['feature', 'mean']].to_string(index=False))

# Identify features to drop
TOP_UNSTABLE_N = 30 if severity in ['SIGNIFICANT', 'SEVERE'] else 20 if severity == 'MODERATE' else 10
unstable_features = fi_avg.head(TOP_UNSTABLE_N)['feature'].tolist()

print(f'\n RECOMMENDATION:')
print(f'  Severity: {severity}')
print(f'  Drop top {TOP_UNSTABLE_N} unstable features')
print(f'  Apply sample weighting in base model retraining')

# Save adversarial probabilities for later sample weighting
adv_probs_train = oof_adv[:len(train)]  # Probability of being "test-like"
adv_probs_test = oof_adv[len(train):]

pd.DataFrame({
    'SK_ID_CURR': train['SK_ID_CURR'].values,
    'adv_prob': adv_probs_train,
}).to_parquet(cfg.OUTPUT_DIR / 'adversarial_probs_train.parquet', index=False)

pd.DataFrame({
    'SK_ID_CURR': test['SK_ID_CURR'].values,
    'adv_prob': adv_probs_test,
}).to_parquet(cfg.OUTPUT_DIR / 'adversarial_probs_test.parquet', index=False)

# Save feature stability info
fi_avg.to_csv(cfg.OUTPUT_DIR / 'adversarial_feature_importance.csv', index=False)

# Save unstable features list
with open(cfg.OUTPUT_DIR / 'unstable_features.txt', 'w') as f:
    for feat in unstable_features:
        f.write(f'{feat}\n')

print(f'\nAdversarial validation done. Time: {(time.time()-start_time)/60:.1f} min')
print(f'\nNEXT: Check the unstable features list. Decision time for Phase A2.')

Loading V12 dataset...
Train: (307507, 1055), Test: (48744, 1054)
Adversarial dataset: (356251, 1053), positive rate: 0.137

Training adversarial classifier...
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1988]	valid's auc: 0.99832
Fold 1/5 | adv AUC: 0.99832 | best_iter: 1988 | time: 565.7s
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1992]	valid's auc: 0.998215
Fold 2/5 | adv AUC: 0.99822 | best_iter: 1992 | time: 530.1s
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid's auc: 0.998359
Fold 3/5 | adv AUC: 0.99836 | best_iter: 2000 | time: 517.0s
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid's auc: 0.998059
Fold 4/5 | adv AUC: 0.99806 | best_iter: 2000 | time: 519.3s
Training until validation scores don't im

In [ ]:
# ========== PHASE A2: DROP UNSTABLE FEATURES ==========
import numpy as np
import pandas as pd
import gc

# Read unstable features list
with open(cfg.OUTPUT_DIR / 'unstable_features.txt', 'r') as f:
    unstable_features = [line.strip() for line in f.readlines()]

print(f'Unstable features to drop: {len(unstable_features)}')
print('Top 10:')
for f in unstable_features[:10]:
    print(f'  - {f}')

# Load V12
train = pd.read_parquet('train_merged_v12.parquet')
test = pd.read_parquet('test_merged_v12.parquet')

# Drop unstable features (only those that exist)
features_to_drop = [f for f in unstable_features if f in train.columns]
print(f'\nDropping {len(features_to_drop)} features that exist in V12')

# Protect critical features (never drop these regardless of adv importance)
PROTECT = [
    'EXT_MEAN', 'EXT_MEDIAN', 'TARGET_NN500_MEAN',
    'PREDICTED_EXT_SOURCE_1', 'PREDICTED_EXT_SOURCE_3',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_BIRTH', 'CREDIT_TERM', 'AMT_CREDIT', 'AMT_INCOME_TOTAL',
    'ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_EDUCATION_TYPE',
    'CODE_GENDER', 'NAME_FAMILY_STATUS',
]
features_to_drop = [f for f in features_to_drop if f not in PROTECT]
print(f'After protecting critical: drop {len(features_to_drop)} features')

train_v14 = train.drop(columns=features_to_drop)
test_v14 = test.drop(columns=features_to_drop)

print(f'\nV14: train={train_v14.shape}, test={test_v14.shape}')

# Save
train_v14.to_parquet('train_merged_v14.parquet', index=False)
test_v14.to_parquet('test_merged_v14.parquet', index=False)

print('V14 saved.')

# Also save sample weights based on adversarial probability
adv_train = pd.read_parquet(cfg.OUTPUT_DIR / 'adversarial_probs_train.parquet')

# Sample weight: higher weight for train samples that look "test-like"
# Formula: w = 1 / (1 - p) clamped, normalized
adv_probs = adv_train['adv_prob'].values
sample_weights = adv_probs / (1 - adv_probs + 0.01)  # Inverse propensity
sample_weights = np.clip(sample_weights, 0.1, 10)  # Clip extreme values
sample_weights = sample_weights / sample_weights.mean()  # Normalize to mean=1

print(f'\nSample weights:')
print(f'  Min:    {sample_weights.min():.3f}')
print(f'  Max:    {sample_weights.max():.3f}')
print(f'  Mean:   {sample_weights.mean():.3f}')
print(f'  Median: {np.median(sample_weights):.3f}')

np.save(cfg.OUTPUT_DIR / 'sample_weights.npy', sample_weights)
print('Sample weights saved.')

del train, test
gc.collect()

Unstable features to drop: 30
Top 10:
  - bb_avg_month
  - MISSING_COUNT
  - bb_max_writeoff_2year
  - AMT_REQ_CREDIT_BUREAU_MON
  - bur_active_total_annuity
  - CREDIT_ANNUITY_RATIO
  - bur_max_enddate
  - CREDIT_TERM
  - BUR_TYPE_DAYS_CREDIT_mean_Consumer_credit
  - OCCUPATION_TYPE_NULL

Dropping 30 features that exist in V12
After protecting critical: drop 27 features

V14: train=(307507, 1028), test=(48744, 1027)
V14 saved.

Sample weights:
  Min:    0.786
  Max:    78.640
  Mean:   1.000
  Median: 0.786
Sample weights saved.


419

## Section 6 - Final V15 construction

This final stage creates row-level risk-score features from previous-application records and bureau records, aggregates those scores to applicant level, merges them into the V14 feature representation, and produces the final V15 matrix.

These row-level score features are treated as leakage-sensitive features and should be generated with train-only or out-of-fold logic when used for validation.

Optional heavy cell: do not run during demo unless local artifacts and sufficient runtime are available.


### **V15**

In [ ]:
# ========== PHASE S1: TRAIN ON EACH PREVIOUS APPLICATION ==========
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import time, gc, warnings
warnings.filterwarnings('ignore')

start_time = time.time()

# Load raw previous_application table
print('Loading raw previous_application...')
prev_app = pd.read_csv('data/raw/previous_application.csv')
application_train = pd.read_csv('data/raw/application_train.csv')
application_test = pd.read_csv('data/raw/application_test.csv')

print(f'Previous_app rows: {len(prev_app):,}')
print(f'Application train: {len(application_train):,}')
print(f'Application test:  {len(application_test):,}')

# Build full SK_ID_CURR → TARGET mapping
# Test rows have no TARGET → exclude from training
train_target_map = application_train.set_index('SK_ID_CURR')['TARGET']
test_ids = set(application_test['SK_ID_CURR'].values)

# Inherit TARGET from current application
prev_app['TARGET'] = prev_app['SK_ID_CURR'].map(train_target_map)
prev_app['is_test'] = prev_app['SK_ID_CURR'].isin(test_ids).astype(int)

# Split: train rows (has TARGET) + test rows (no TARGET, just predict)
prev_train = prev_app[prev_app['TARGET'].notna()].copy()
prev_test = prev_app[prev_app['is_test'] == 1].copy()

print(f'\nPrevious_app for training (has TARGET): {len(prev_train):,}')
print(f'Previous_app for prediction (test rows): {len(prev_test):,}')
print(f'TARGET rate in prev_train: {prev_train["TARGET"].mean():.4f}')

# === Feature Engineering trên Previous Application ===
print('\n[1/3] Feature engineering on previous_application...')

def engineer_prev_features(df):
    """Engineer features cho mỗi previous_application row."""
    df = df.copy()

    # Time-based features
    if 'DAYS_DECISION' in df.columns:
        df['DAYS_DECISION_abs'] = df['DAYS_DECISION'].abs()
        df['YEAR_DECISION'] = (df['DAYS_DECISION'] / -365.25).clip(0, 30)

    if 'DAYS_FIRST_DRAWING' in df.columns:
        df['DAYS_FIRST_DRAWING'] = df['DAYS_FIRST_DRAWING'].replace(365243, np.nan)
    if 'DAYS_FIRST_DUE' in df.columns:
        df['DAYS_FIRST_DUE'] = df['DAYS_FIRST_DUE'].replace(365243, np.nan)
    if 'DAYS_LAST_DUE_1ST_VERSION' in df.columns:
        df['DAYS_LAST_DUE_1ST_VERSION'] = df['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan)
    if 'DAYS_LAST_DUE' in df.columns:
        df['DAYS_LAST_DUE'] = df['DAYS_LAST_DUE'].replace(365243, np.nan)
    if 'DAYS_TERMINATION' in df.columns:
        df['DAYS_TERMINATION'] = df['DAYS_TERMINATION'].replace(365243, np.nan)

    # Ratios
    eps = 1e-5
    if 'AMT_CREDIT' in df.columns and 'AMT_APPLICATION' in df.columns:
        df['APP_CREDIT_RATIO'] = df['AMT_APPLICATION'] / (df['AMT_CREDIT'] + eps)
        df['APP_CREDIT_DIFF'] = df['AMT_APPLICATION'] - df['AMT_CREDIT']

    if 'AMT_CREDIT' in df.columns and 'AMT_ANNUITY' in df.columns:
        df['CREDIT_ANNUITY_RATIO_PREV'] = df['AMT_CREDIT'] / (df['AMT_ANNUITY'] + eps)

    if 'AMT_DOWN_PAYMENT' in df.columns and 'AMT_CREDIT' in df.columns:
        df['DOWN_PAYMENT_RATIO'] = df['AMT_DOWN_PAYMENT'] / (df['AMT_CREDIT'] + eps)

    if 'CNT_PAYMENT' in df.columns and 'AMT_CREDIT' in df.columns:
        df['CREDIT_PER_PAYMENT'] = df['AMT_CREDIT'] / (df['CNT_PAYMENT'] + eps)

    # Loan duration
    if 'DAYS_LAST_DUE' in df.columns and 'DAYS_DECISION' in df.columns:
        df['LOAN_DURATION'] = df['DAYS_LAST_DUE'] - df['DAYS_DECISION']

    return df

prev_train = engineer_prev_features(prev_train)
prev_test = engineer_prev_features(prev_test)

print(f'Engineered features done. Columns: {prev_train.shape[1]}')

# === Select features for training ===
EXCLUDE_COLS = ['SK_ID_CURR', 'SK_ID_PREV', 'TARGET', 'is_test']
feature_cols = [c for c in prev_train.columns if c not in EXCLUDE_COLS]

# Encode categoricals
for col in feature_cols:
    if prev_train[col].dtype == 'object':
        combined = pd.concat([prev_train[col].astype(str), prev_test[col].astype(str)])
        codes, _ = pd.factorize(combined)
        prev_train[col] = codes[:len(prev_train)].astype(np.int32)
        prev_test[col] = codes[len(prev_train):].astype(np.int32)

print(f'Features for training: {len(feature_cols)}')

# === Training: CRITICAL — GroupKFold by SK_ID_CURR ===
# Same SK_ID_CURR phải trong cùng fold (avoid leakage)
print('\n[2/3] Training LGBM với GroupKFold by SK_ID_CURR...')

LGB_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 32, 'max_depth': 7,
    'min_child_samples': 200, 'min_child_weight': 10,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'colsample_bytree': 0.7, 'subsample': 0.8, 'subsample_freq': 1,
    'max_bin': 255, 'verbosity': -1, 'random_state': 42, 'n_jobs': -1,
}

N_FOLDS = 5
y_prev = prev_train['TARGET'].astype(int).values
groups = prev_train['SK_ID_CURR'].values
X_prev = prev_train[feature_cols]
X_prev_test = prev_test[feature_cols]

# GroupKFold to prevent SK_ID_CURR leakage
gkf = GroupKFold(n_splits=N_FOLDS)

oof_prev_scores = np.zeros(len(prev_train))
test_prev_scores = np.zeros(len(prev_test))
fold_aucs = []

for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(X_prev, y_prev, groups)):
    fold_start = time.time()
    X_tr, X_val = X_prev.iloc[tr_idx], X_prev.iloc[val_idx]
    y_tr, y_val = y_prev[tr_idx], y_prev[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)

    model = lgb.train(
        LGB_PARAMS, dtrain, num_boost_round=2000,
        valid_sets=[dval], valid_names=['valid'],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
    )

    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds = model.predict(X_prev_test, num_iteration=model.best_iteration)

    oof_prev_scores[val_idx] = val_preds
    test_prev_scores += test_preds / N_FOLDS

    val_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(val_auc)
    print(f'  Fold {fold_idx+1}/5 | val AUC: {val_auc:.5f} | best_iter: {model.best_iteration} | time: {time.time()-fold_start:.1f}s')
    del model, dtrain, dval
    gc.collect()

print(f'\nPrev-level CV AUC: {np.mean(fold_aucs):.5f} (note: row-level, not customer-level)')

# === Aggregate predictions back to SK_ID_CURR ===
print('\n[3/3] Aggregating predictions back to SK_ID_CURR...')

prev_train['prev_score'] = oof_prev_scores
prev_test['prev_score'] = test_prev_scores

# Combine train + test scores
all_prev = pd.concat([prev_train, prev_test], ignore_index=True)

# Aggregate by SK_ID_CURR
prev_agg_scores = all_prev.groupby('SK_ID_CURR').agg({
    'prev_score': ['mean', 'max', 'min', 'std', 'count'],
}).reset_index()

prev_agg_scores.columns = [
    'SK_ID_CURR',
    'PREV_SCORE_MEAN', 'PREV_SCORE_MAX', 'PREV_SCORE_MIN',
    'PREV_SCORE_STD', 'PREV_SCORE_COUNT'
]

# Additional: most recent N predictions
print('  Computing recent predictions aggregations...')

# Recent 3 (last 3 prev_app by DAYS_DECISION descending = most recent)
all_prev_sorted = all_prev.sort_values(['SK_ID_CURR', 'DAYS_DECISION'], ascending=[True, False])
recent_3 = all_prev_sorted.groupby('SK_ID_CURR').head(3)
recent_3_agg = recent_3.groupby('SK_ID_CURR')['prev_score'].agg(['mean', 'max']).reset_index()
recent_3_agg.columns = ['SK_ID_CURR', 'PREV_SCORE_RECENT3_MEAN', 'PREV_SCORE_RECENT3_MAX']

# Recent 1 (last prev_app)
last_1 = all_prev_sorted.groupby('SK_ID_CURR').head(1)
last_1_agg = last_1[['SK_ID_CURR', 'prev_score']].rename(columns={'prev_score': 'PREV_SCORE_LAST'})

# Merge all
prev_features = prev_agg_scores.merge(recent_3_agg, on='SK_ID_CURR', how='left')
prev_features = prev_features.merge(last_1_agg, on='SK_ID_CURR', how='left')

print(f'\nTotal new features from prev_app: {prev_features.shape[1] - 1}')
print(f'Features: {[c for c in prev_features.columns if c != "SK_ID_CURR"]}')

# Save
prev_features.to_parquet(cfg.OUTPUT_DIR / 'prev_app_target_features.parquet', index=False)

# Cleanup
del prev_train, prev_test, all_prev, prev_app, application_train, application_test
gc.collect()

print(f'\nPhase S1 done. Time: {(time.time()-start_time)/60:.1f} min')
print(f'Features saved to prev_app_target_features.parquet')

Loading raw previous_application...
Previous_app rows: 1,670,214
Application train: 307,511
Application test:  48,744

Previous_app for training (has TARGET): 1,413,701
Previous_app for prediction (test rows): 256,513
TARGET rate in prev_train: 0.0866

[1/3] Feature engineering on previous_application...
Engineered features done. Columns: 47
Features for training: 43

[2/3] Training LGBM với GroupKFold by SK_ID_CURR...
Training until validation scores don't improve for 100 rounds
[200]	valid's auc: 0.626468
[400]	valid's auc: 0.627693
[600]	valid's auc: 0.627844
Early stopping, best iteration is:
[536]	valid's auc: 0.627879
  Fold 1/5 | val AUC: 0.62788 | best_iter: 536 | time: 80.3s
Training until validation scores don't improve for 100 rounds
[200]	valid's auc: 0.619074
[400]	valid's auc: 0.620655
[600]	valid's auc: 0.621046
Early stopping, best iteration is:
[579]	valid's auc: 0.621076
  Fold 2/5 | val AUC: 0.62108 | best_iter: 579 | time: 90.5s
Training until validation scores don'

In [ ]:
# ========== PHASE S2: TRAIN ON EACH BUREAU RECORD ==========
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import time, gc

start_time = time.time()

# Load raw bureau table
print('Loading raw bureau...')
bureau = pd.read_csv('data/raw/bureau.csv')
bureau_balance = pd.read_csv('data/raw/bureau_balance.csv')
application_train = pd.read_csv('data/raw/application_train.csv')
application_test = pd.read_csv('data/raw/application_test.csv')

print(f'Bureau rows: {len(bureau):,}')

# Aggregate bureau_balance to bureau level
print('Aggregating bureau_balance to bureau level...')
bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE': ['count', 'min', 'max', 'mean'],
}).reset_index()
bb_agg.columns = ['SK_ID_BUREAU', 'bb_count', 'bb_first', 'bb_last', 'bb_mean_month']

# Status counts (C=Closed, X=Unknown, 0-5=DPD)
bb_status = pd.crosstab(bureau_balance['SK_ID_BUREAU'], bureau_balance['STATUS'])
bb_status.columns = [f'bb_status_{c}' for c in bb_status.columns]
bb_status = bb_status.reset_index()

bb_features = bb_agg.merge(bb_status, on='SK_ID_BUREAU', how='left')
print(f'Bureau_balance features: {bb_features.shape[1] - 1}')

# Merge to bureau
bureau = bureau.merge(bb_features, on='SK_ID_BUREAU', how='left')

# Build TARGET mapping
train_target_map = application_train.set_index('SK_ID_CURR')['TARGET']
test_ids = set(application_test['SK_ID_CURR'].values)

bureau['TARGET'] = bureau['SK_ID_CURR'].map(train_target_map)
bureau['is_test'] = bureau['SK_ID_CURR'].isin(test_ids).astype(int)

bureau_train = bureau[bureau['TARGET'].notna()].copy()
bureau_test = bureau[bureau['is_test'] == 1].copy()

print(f'\nBureau for training: {len(bureau_train):,}')
print(f'Bureau for prediction: {len(bureau_test):,}')
print(f'TARGET rate: {bureau_train["TARGET"].mean():.4f}')

# === Feature Engineering ===
print('\n[1/3] Feature engineering on bureau...')

def engineer_bureau_features(df):
    """Engineer features cho mỗi bureau row."""
    df = df.copy()
    eps = 1e-5

    # Ratios
    if 'AMT_CREDIT_SUM' in df.columns and 'AMT_CREDIT_SUM_DEBT' in df.columns:
        df['BUR_DEBT_RATIO'] = df['AMT_CREDIT_SUM_DEBT'] / (df['AMT_CREDIT_SUM'] + eps)

    if 'AMT_CREDIT_SUM' in df.columns and 'AMT_CREDIT_SUM_LIMIT' in df.columns:
        df['BUR_LIMIT_RATIO'] = df['AMT_CREDIT_SUM_LIMIT'] / (df['AMT_CREDIT_SUM'] + eps)

    if 'AMT_CREDIT_SUM' in df.columns and 'AMT_ANNUITY' in df.columns:
        df['BUR_CREDIT_ANNUITY_RATIO'] = df['AMT_CREDIT_SUM'] / (df['AMT_ANNUITY'] + eps)

    # Time features
    if 'DAYS_CREDIT' in df.columns:
        df['DAYS_CREDIT_abs'] = df['DAYS_CREDIT'].abs()
        df['YEAR_CREDIT'] = (df['DAYS_CREDIT'] / -365.25).clip(0, 30)

    if 'DAYS_CREDIT_ENDDATE' in df.columns and 'DAYS_CREDIT' in df.columns:
        df['BUR_DURATION'] = df['DAYS_CREDIT_ENDDATE'] - df['DAYS_CREDIT']

    if 'DAYS_ENDDATE_FACT' in df.columns and 'DAYS_CREDIT_ENDDATE' in df.columns:
        df['BUR_PROLONGATION'] = df['DAYS_ENDDATE_FACT'] - df['DAYS_CREDIT_ENDDATE']

    return df

bureau_train = engineer_bureau_features(bureau_train)
bureau_test = engineer_bureau_features(bureau_test)

print(f'Engineered features. Columns: {bureau_train.shape[1]}')

# === Select features ===
EXCLUDE_COLS = ['SK_ID_CURR', 'SK_ID_BUREAU', 'TARGET', 'is_test']
feature_cols = [c for c in bureau_train.columns if c not in EXCLUDE_COLS]

for col in feature_cols:
    if bureau_train[col].dtype == 'object':
        combined = pd.concat([bureau_train[col].astype(str), bureau_test[col].astype(str)])
        codes, _ = pd.factorize(combined)
        bureau_train[col] = codes[:len(bureau_train)].astype(np.int32)
        bureau_test[col] = codes[len(bureau_train):].astype(np.int32)

print(f'Features for training: {len(feature_cols)}')

# === Training với GroupKFold ===
print('\n[2/3] Training LGBM với GroupKFold by SK_ID_CURR...')

LGB_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 32, 'max_depth': 7,
    'min_child_samples': 200, 'min_child_weight': 10,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'colsample_bytree': 0.7, 'subsample': 0.8, 'subsample_freq': 1,
    'max_bin': 255, 'verbosity': -1, 'random_state': 42, 'n_jobs': -1,
}

N_FOLDS = 5
y_bur = bureau_train['TARGET'].astype(int).values
groups = bureau_train['SK_ID_CURR'].values
X_bur = bureau_train[feature_cols]
X_bur_test = bureau_test[feature_cols]

gkf = GroupKFold(n_splits=N_FOLDS)

oof_bur_scores = np.zeros(len(bureau_train))
test_bur_scores = np.zeros(len(bureau_test))
fold_aucs = []

for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(X_bur, y_bur, groups)):
    fold_start = time.time()
    X_tr, X_val = X_bur.iloc[tr_idx], X_bur.iloc[val_idx]
    y_tr, y_val = y_bur[tr_idx], y_bur[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)

    model = lgb.train(
        LGB_PARAMS, dtrain, num_boost_round=2000,
        valid_sets=[dval], valid_names=['valid'],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
    )

    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds = model.predict(X_bur_test, num_iteration=model.best_iteration)

    oof_bur_scores[val_idx] = val_preds
    test_bur_scores += test_preds / N_FOLDS

    val_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(val_auc)
    print(f'  Fold {fold_idx+1}/5 | val AUC: {val_auc:.5f} | best_iter: {model.best_iteration} | time: {time.time()-fold_start:.1f}s')
    del model, dtrain, dval
    gc.collect()

print(f'\nBureau-level CV AUC: {np.mean(fold_aucs):.5f}')

# === Aggregate ===
print('\n[3/3] Aggregating predictions back to SK_ID_CURR...')

bureau_train['bur_score'] = oof_bur_scores
bureau_test['bur_score'] = test_bur_scores

all_bur = pd.concat([bureau_train, bureau_test], ignore_index=True)

bur_agg_scores = all_bur.groupby('SK_ID_CURR').agg({
    'bur_score': ['mean', 'max', 'min', 'std', 'count'],
}).reset_index()

bur_agg_scores.columns = [
    'SK_ID_CURR',
    'BUR_SCORE_MEAN', 'BUR_SCORE_MAX', 'BUR_SCORE_MIN',
    'BUR_SCORE_STD', 'BUR_SCORE_COUNT'
]

# Active vs Closed
if 'CREDIT_ACTIVE' in all_bur.columns:
    # Need original CREDIT_ACTIVE (string)
    # Already encoded — use code 0 (Active typically first)
    pass

# Recent (last 12 months)
all_bur_recent = all_bur[all_bur['DAYS_CREDIT'] >= -365]
if len(all_bur_recent) > 0:
    bur_recent_agg = all_bur_recent.groupby('SK_ID_CURR')['bur_score'].agg(['mean', 'max']).reset_index()
    bur_recent_agg.columns = ['SK_ID_CURR', 'BUR_SCORE_RECENT12M_MEAN', 'BUR_SCORE_RECENT12M_MAX']
    bur_agg_scores = bur_agg_scores.merge(bur_recent_agg, on='SK_ID_CURR', how='left')

print(f'\nTotal new features from bureau: {bur_agg_scores.shape[1] - 1}')

bur_agg_scores.to_parquet(cfg.OUTPUT_DIR / 'bureau_target_features.parquet', index=False)

del bureau_train, bureau_test, all_bur, bureau, bureau_balance, application_train, application_test, bb_features
gc.collect()

print(f'\nPhase S2 done. Time: {(time.time()-start_time)/60:.1f} min')

Loading raw bureau...
Bureau rows: 1,716,428
Aggregating bureau_balance to bureau level...
Bureau_balance features: 12

Bureau for training: 1,465,325
Bureau for prediction: 251,103
TARGET rate: 0.0782

[1/3] Feature engineering on bureau...
Engineered features. Columns: 38
Features for training: 34

[2/3] Training LGBM với GroupKFold by SK_ID_CURR...
Training until validation scores don't improve for 100 rounds
[200]	valid's auc: 0.612841
[400]	valid's auc: 0.614336
[600]	valid's auc: 0.614682
Early stopping, best iteration is:
[592]	valid's auc: 0.614709
  Fold 1/5 | val AUC: 0.61471 | best_iter: 592 | time: 61.1s
Training until validation scores don't improve for 100 rounds
[200]	valid's auc: 0.622276
[400]	valid's auc: 0.624173
[600]	valid's auc: 0.624384
Early stopping, best iteration is:
[573]	valid's auc: 0.624501
  Fold 2/5 | val AUC: 0.62450 | best_iter: 573 | time: 66.6s
Training until validation scores don't improve for 100 rounds
[200]	valid's auc: 0.61256
[400]	valid's auc

In [ ]:
# ========== PHASE S3: BUILD V15 + TRAIN LGBM ==========
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import time, gc

start_time = time.time()

# Load V14 + new features
print('Loading V14 + target features...')
train = pd.read_parquet('train_merged_v14.parquet')
test = pd.read_parquet('test_merged_v14.parquet')

prev_features = pd.read_parquet(cfg.OUTPUT_DIR / 'prev_app_target_features.parquet')
bur_features = pd.read_parquet(cfg.OUTPUT_DIR / 'bureau_target_features.parquet')

print(f'V14: train={train.shape}, test={test.shape}')
print(f'Prev features: {prev_features.shape}')
print(f'Bureau features: {bur_features.shape}')

# Merge
train_v15 = train.merge(prev_features, on='SK_ID_CURR', how='left')
train_v15 = train_v15.merge(bur_features, on='SK_ID_CURR', how='left')

test_v15 = test.merge(prev_features, on='SK_ID_CURR', how='left')
test_v15 = test_v15.merge(bur_features, on='SK_ID_CURR', how='left')

print(f'\nV15: train={train_v15.shape}, test={test_v15.shape}')

new_features = [c for c in prev_features.columns if c != 'SK_ID_CURR'] + \
               [c for c in bur_features.columns if c != 'SK_ID_CURR']
print(f'New target-encoded features added: {len(new_features)}')
print(f'  Prev features: {[c for c in prev_features.columns if c != "SK_ID_CURR"]}')
print(f'  Bureau features: {[c for c in bur_features.columns if c != "SK_ID_CURR"]}')

# Save V15
train_v15.to_parquet('train_merged_v15.parquet', index=False)
test_v15.to_parquet('test_merged_v15.parquet', index=False)

# === Train LGBM V15 multi-seed ===
EXCLUDE_COLS = ['SK_ID_CURR', cfg.TARGET, 'index']
feature_cols = [c for c in train_v15.columns if c not in EXCLUDE_COLS]

for col in feature_cols:
    if train_v15[col].dtype == 'object':
        combined = pd.concat([train_v15[col].astype(str), test_v15[col].astype(str)])
        codes, _ = pd.factorize(combined)
        train_v15[col] = codes[:len(train_v15)].astype(np.int32)
        test_v15[col] = codes[len(train_v15):].astype(np.int32)

FORCE_CATEGORICAL = [
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'WEEKDAY_APPR_PROCESS_START',
    'HOUR_APPR_PROCESS_START', 'BURO_CLU_LABEL', 'PREV_CLU_LABEL',
]
categorical_features = [c for c in feature_cols if c in FORCE_CATEGORICAL]

y = train_v15[cfg.TARGET].astype(int).values
X = train_v15[feature_cols]
X_test = test_v15[feature_cols]

print(f'\nV15 total features: {len(feature_cols)}')

LGB_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'learning_rate': 0.02, 'num_leaves': 24, 'max_depth': 6,
    'min_child_samples': 100, 'min_child_weight': 10,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'colsample_bytree': 0.7, 'subsample': 0.8, 'subsample_freq': 1,
    'max_bin': 255, 'verbosity': -1, 'n_jobs': -1,
}

N_FOLDS = 5
SEEDS = [42, 7, 99]

v15_oofs = {}
v15_tests = {}

for seed in SEEDS:
    print(f'\n{"="*70}')
    print(f'TRAINING V15 — seed={seed}')
    print(f'{"="*70}')

    params = dict(LGB_PARAMS)
    params['random_state'] = seed

    oof_seed = np.zeros(len(train_v15))
    test_seed = np.zeros(len(test_v15))
    fold_aucs = []
    fi_list = []

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        fold_start = time.time()
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_features)
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_features)

        model = lgb.train(
            params, dtrain, num_boost_round=5000,
            valid_sets=[dval], valid_names=['valid'],
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)],
        )

        val_preds = model.predict(X_val, num_iteration=model.best_iteration)
        test_fold_preds = model.predict(X_test, num_iteration=model.best_iteration)

        oof_seed[val_idx] = val_preds
        test_seed += test_fold_preds / N_FOLDS

        val_auc = roc_auc_score(y_val, val_preds)
        fold_aucs.append(val_auc)

        if seed == 42:
            fi_list.append(pd.DataFrame({
                'feature': feature_cols,
                'importance': model.feature_importance(importance_type='gain'),
            }))

        print(f'  Fold {fold_idx+1}/5 | val AUC: {val_auc:.5f} | best_iter: {model.best_iteration} | time: {time.time()-fold_start:.1f}s')
        del model, dtrain, dval
        gc.collect()

    seed_auc = roc_auc_score(y, oof_seed)
    print(f'  Seed {seed} OOF: {seed_auc:.5f}')

    v15_oofs[seed] = oof_seed
    v15_tests[seed] = test_seed

# Multi-seed average
oof_v15_ms = np.mean(list(v15_oofs.values()), axis=0)
test_v15_ms = np.mean(list(v15_tests.values()), axis=0)
auc_v15_ms = roc_auc_score(y, oof_v15_ms)

print(f'\n{"="*70}')
print(f'LGBM V15 MULTI-SEED RESULTS:')
print(f'{"="*70}')
for seed, oof in v15_oofs.items():
    print(f'  Seed {seed}: {roc_auc_score(y, oof):.5f}')
print(f'  Multi-seed avg:  {auc_v15_ms:.5f}')
print(f'  Comparison:')
print(f'    V12 multi-seed: 0.79854')
print(f'    V14 (dropped):  TBD')
print(f'    V15 (NEW):       {auc_v15_ms:.5f}')
print(f'    Gain:            {auc_v15_ms - 0.79854:+.5f}')

# Check ranks of NEW features
if fi_list:
    fi_avg = pd.concat(fi_list).groupby('feature')['importance'].mean().reset_index()
    fi_avg = fi_avg.sort_values('importance', ascending=False).reset_index(drop=True)

    print(f'\nNew target-encoded features ranks:')
    for f in new_features:
        rank = fi_avg[fi_avg['feature'] == f].index
        if len(rank) > 0:
            imp = fi_avg.loc[rank[0], 'importance']
            print(f'  rank {rank[0]+1:4d}: {f:35s} | imp: {imp:.0f}')

    print(f'\nTop 20 V15 features overall:')
    print(fi_avg.head(20).to_string(index=False))

# Save
sk_ids_train = train_v15['SK_ID_CURR'].values
sk_ids_test = test_v15['SK_ID_CURR'].values

pd.DataFrame({
    'SK_ID_CURR': sk_ids_train, cfg.TARGET: y, 'oof_lgb': oof_v15_ms,
}).to_parquet(cfg.OUTPUT_DIR / 'oof_lgb_v15_multiseed.parquet', index=False)

pd.DataFrame({
    'SK_ID_CURR': sk_ids_test, 'TARGET': test_v15_ms,
}).to_csv(cfg.OUTPUT_DIR / 'submission_lgb_v15_multiseed.csv', index=False)

if fi_list:
    fi_avg.to_csv(cfg.OUTPUT_DIR / 'feature_importance_lgb_v15.csv', index=False)

print(f'\nPhase S3 done. Time: {(time.time()-start_time)/60:.1f} min')

del train, test, train_v15, test_v15
gc.collect()

Loading V14 + target features...
V14: train=(307507, 1028), test=(48744, 1027)
Prev features: (338857, 9)
Bureau features: (305811, 8)

V15: train=(307507, 1043), test=(48744, 1042)
New target-encoded features added: 15
  Prev features: ['PREV_SCORE_MEAN', 'PREV_SCORE_MAX', 'PREV_SCORE_MIN', 'PREV_SCORE_STD', 'PREV_SCORE_COUNT', 'PREV_SCORE_RECENT3_MEAN', 'PREV_SCORE_RECENT3_MAX', 'PREV_SCORE_LAST']
  Bureau features: ['BUR_SCORE_MEAN', 'BUR_SCORE_MAX', 'BUR_SCORE_MIN', 'BUR_SCORE_STD', 'BUR_SCORE_COUNT', 'BUR_SCORE_RECENT12M_MEAN', 'BUR_SCORE_RECENT12M_MAX']

V15 total features: 1041

TRAINING V15 — seed=42
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1534]	valid's auc: 0.793869
  Fold 1/5 | val AUC: 0.79387 | best_iter: 1534 | time: 352.0s
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1893]	valid's auc: 0.805047
  Fold 2/5 | val AUC: 0.80505 | best_iter: 1893 | time: 405.4s
Traini

0

## Section 7 - Final V15 summary

V15 has **1,043 columns including `SK_ID_CURR` and `TARGET`**. V15 has **1,041 modelling features** after excluding ID and target columns.

The compact summary below is safe to run locally. It inspects the saved V15 parquet files if they are present; otherwise it prints the thesis-facing expected matrix contract documented from the verified local artifacts.


In [ ]:

from pathlib import Path
import pandas as pd

train_path = Path('processed_train_test/train_merged_v15.parquet')
test_path = Path('processed_train_test/test_merged_v15.parquet')

expected = {
    'train_v15_shape': (307507, 1043),
    'test_v15_shape': (48744, 1042),
    'total_train_columns': 1043,
    'modelling_feature_count': 1041,
    'saved_train_path': str(train_path),
    'saved_test_path': str(test_path),
}

categorical_columns = [
    'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_INCOME_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'NAME_CONTRACT_TYPE',
    'CODE_GENDER',
    'WEEKDAY_APPR_PROCESS_START',
    'HOUR_APPR_PROCESS_START',
    'BURO_CLU_LABEL',
    'PREV_CLU_LABEL',
]

if train_path.exists() and test_path.exists():
    train_v15 = pd.read_parquet(train_path)
    test_v15 = pd.read_parquet(test_path)
    modelling_features = [c for c in train_v15.columns if c not in ['SK_ID_CURR', 'TARGET']]
    present_categoricals = [c for c in categorical_columns if c in train_v15.columns]
    summary = {
        'train_v15_shape': train_v15.shape,
        'test_v15_shape': test_v15.shape,
        'total_train_columns': len(train_v15.columns),
        'modelling_feature_count': len(modelling_features),
        'categorical_columns_present': present_categoricals,
        'saved_train_path': str(train_path),
        'saved_test_path': str(test_path),
    }
else:
    summary = {
        **expected,
        'categorical_columns_expected': categorical_columns,
        'note': 'Local V15 parquet artifacts are not present in this checkout or are gitignored.',
    }

summary


{'train_v15_shape': (307507, 1043),
 'test_v15_shape': (48744, 1042),
 'total_train_columns': 1043,
 'modelling_feature_count': 1041,
 'categorical_columns_present': ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'BURO_CLU_LABEL', 'PREV_CLU_LABEL'],
 'saved_train_path': 'processed_train_test/train_merged_v15.parquet',
 'saved_test_path': 'processed_train_test/test_merged_v15.parquet'}